# Image Studio — Normal v10

**Create images. Edit images. Face/head swap. Upscale.**

Attach your existing **ONE ALL MODELS Dataset**, select a Kaggle T4 GPU and enable Internet for software/tunnel setup. Run the single startup cell, open the printed ComfyUI URL, choose **Image Studio - …** from the workflow menu and press Run. No image starts automatically. Your model dataset, authentication and dataset creator are unchanged. No model download or Dataset creation runs in the studio.

| Workflow | What you do |
|---|---|
| Create - Cinematic | Write your prompt, set width/height and seed, then Run. Defaults to 1216×832 landscape; use multiples of 16. |
| Image Edit | Upload a picture and describe the change. |
| Inpaint - Masked Edit | Upload a picture, right-click → MaskEditor, paint white over the area to replace, then describe the change. |
| Face Swap - Reference Edit | Target is image 1; source identity is image 2. Describe which person changes. |
| Face Swap - Masked | Same inputs, with a painted replacement mask to keep changes local. |
| Head Swap - BFS | Target is Picture 1; identity is Picture 2. This dedicated adapter can replace head and hair. |
| Face Composite - Simple | Transfer source pixels onto a similar-pose target. Choose face indices (0 = leftmost). Simple fallback with possible visible edges. |
| Upscale 2x | Upload, enlarge to twice the dimensions, save. |
| Upscale 4x | The same Real-ESRGAN x4plus workflow as before. |
| Restore SeedVR2 | Upload a still for reconstruction/upscaling; inspect invented detail. |

Prompt, dimensions and seed are ordinary visible nodes. Results save as PNGs with graph/settings metadata, under `/kaggle/working/image-studio/images`. There are no episode presets, cast-approval stages, scripture fields, lettering or branded exports. Older app-managed workflow files are backed up outside the workflow menu. Your source photos and existing output images are not deleted.

The reviewed models are unchanged: Klein 4B distilled generation/reference editing, BFS 4B adapter, Real-ESRGAN and SeedVR2, with the optional simple landmark compositor. Klein starts at four steps/CFG 1; increasing steps is not a proven quality upgrade for this distilled model. Cinematic quality depends on the prompt, scene, references and chosen output; no measured “best model” claim is made. Reference editing and head replacement can change likeness and scene details. Masked edits are generated candidates pasted within your mask/feather, not a separate trained inpainting model. The upscaling graphs are reused unchanged.

Keep the cell running while using ComfyUI, keep the session URL private, and Save Version with outputs before ending the session. The existing commercial model license notices/provenance are preserved. Full GPU inference and T4 quality/speed remain unverified here; validation results are in VALIDATION.md.


In [ ]:
# Single-cell startup. No image generation starts automatically.
from pathlib import Path
import sys, os, subprocess, json, time, urllib.request, hashlib, shutil, re, signal, fcntl
ENABLE_CREATE_EDIT = True
ENABLE_SEEDVR2 = True
ENABLE_MESH_SWAP = True
ENABLE_BFS_HEADSWAP = True
"""Find and verify saved model weights. Never download or publish during startup."""
from pathlib import Path
import hashlib

def prepare_models(assets,roots):
 def digest(path):
  h=hashlib.sha256()
  with open(path,'rb') as f:
   for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
  return h.hexdigest()
 resolved={}
 for asset in assets:
  for root in map(Path,roots):
   if not root.exists():continue
   found=next((p for p in root.rglob(asset['name']) if p.is_file() and p.stat().st_size==asset['bytes'] and digest(p)==asset['sha256']),None)
   if found:resolved[asset['name']]=found;break
 missing=[a['name'] for a in assets if a['name'] not in resolved]
 if missing:raise RuntimeError('Attach your saved ALL MODELS Dataset (or its saved notebook output). Missing: '+', '.join(missing)+'. No model downloads or Dataset uploads were started. Build the all pack once on CPU with Image_Studio_All_Models_Dataset.ipynb.')
 return resolved

"""Replace app-managed menus while preserving old canvases outside the menu."""
from pathlib import Path
import json,hashlib,shutil

def install_workflows(directory,payload,archive):
 directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)
 expected={'Image Studio - '+name+'.json' for name in payload}
 for old in directory.glob('Image Studio - *.json'):
  # Preserve all prior managed canvases; replace only this application's files.
  blob=old.read_bytes();dest=Path(archive)/(old.stem+'-'+hashlib.sha256(blob).hexdigest()[:10]+'.json');dest.parent.mkdir(parents=True,exist_ok=True)
  if not dest.exists():shutil.copyfile(old,dest)
  if old.name not in expected:old.unlink()
 for name,wf in payload.items():(directory/('Image Studio - '+name+'.json')).write_text(json.dumps(wf,indent=2))

FACE_NODE_SOURCE = '"""CPU landmark mesh compositor. No identity encoder or learned face-swap weights."""\nimport os,json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport cv2\nimport folder_paths\n\ndef rgb(t):\n if t.ndim!=4 or t.shape[0]!=1:raise ValueError(\'Use one image per input.\')\n return (t[0].cpu().numpy().clip(0,1)*255).round().astype(np.uint8)\ndef detect(image):\n import mediapipe as mp\n task=Path(os.environ.get(\'STUDIO_LANDMARK_MODEL\',str(Path(folder_paths.models_dir)/\'mediapipe/face_landmarker.task\')))\n if not task.is_file():raise ValueError(\'Attach the commercial finish pack: face_landmarker.task is missing.\')\n options=mp.tasks.vision.FaceLandmarkerOptions(base_options=mp.tasks.BaseOptions(model_asset_path=str(task)),num_faces=10,output_face_blendshapes=False,output_facial_transformation_matrixes=False)\n with mp.tasks.vision.FaceLandmarker.create_from_options(options) as model:\n  result=model.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=np.ascontiguousarray(image)))\n h,w=image.shape[:2]\n points=[np.array([(min(w-1,max(0,p.x*w)),min(h-1,max(0,p.y*h))) for p in face[:468]],np.float32) for face in result.face_landmarks]\n return sorted(points,key=lambda p:p[:,0].mean())\ndef composite(source,target,source_index=0,target_index=0,blend_mode="Feather"):\n sp,tp=detect(source),detect(target)\n if not 0<=source_index<len(sp) or not 0<=target_index<len(tp):raise ValueError(f\'Faces found: source={len(sp)}, target={len(tp)}; indices start at 0, left to right.\')\n a,b=sp[source_index],tp[target_index];h,w=target.shape[:2];warp=np.zeros_like(target)\n # Triangulation derives only from landmark positions; never loads an embedding model.\n hull=cv2.convexHull(b).reshape(-1,2);sub=cv2.Subdiv2D((0,0,w,h))\n for p in b:sub.insert(tuple(map(float,p)))\n for tri in sub.getTriangleList():\n  pts=np.array(tri).reshape(3,2)\n  if np.any(pts[:,0]<0) or np.any(pts[:,0]>=w) or np.any(pts[:,1]<0) or np.any(pts[:,1]>=h):continue\n  ids=[int(np.argmin(np.linalg.norm(b-p,axis=1))) for p in pts]\n  if len(set(ids))<3:continue\n  src=a[ids];dst=b[ids];sx,sy,sw,sh=cv2.boundingRect(src);x,y,tw,th=cv2.boundingRect(dst)\n  if min(sw,sh,tw,th)<1:continue\n  mat=cv2.getAffineTransform((src-[sx,sy]).astype(np.float32),(dst-[x,y]).astype(np.float32))\n  tile=cv2.warpAffine(source[sy:sy+sh,sx:sx+sw],mat,(tw,th),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_REFLECT_101)\n  mask=np.zeros((th,tw),np.uint8);cv2.fillConvexPoly(mask,np.int32(dst-[x,y]),255)\n  view=warp[y:y+th,x:x+tw];view[mask>0]=tile[mask>0]\n mask=np.zeros((h,w),np.uint8);cv2.fillConvexPoly(mask,np.int32(hull),255)\n # Keep edge inside the face hull and use Poisson blending for lighting continuity.\n mask=cv2.erode(mask,np.ones((3,3),np.uint8))\n x,y,bw,bh=cv2.boundingRect(mask)\n if not np.any(mask):raise ValueError(\'Face is too small to composite.\')\n if blend_mode==\'Poisson\':\n  out=cv2.seamlessClone(warp,target,mask.copy(),(x+bw//2,y+bh//2),cv2.NORMAL_CLONE)\n else:\n  # Preserve the transferred features; Poisson blending can weaken identity contrast.\n  alpha=cv2.distanceTransform(mask,cv2.DIST_L2,3);alpha=np.clip(alpha/max(2,min(bw,bh)*0.04),0,1)[...,None]\n  out=(warp*alpha+target*(1-alpha)).round().astype(np.uint8)\n out[mask==0]=target[mask==0]\n return out,dict(method=\'mediapipe-mesh-pixel-composite\',source_faces=len(sp),target_faces=len(tp),source_index=source_index,target_index=target_index,blend_mode=blend_mode,limitations=\'Use similar pose/expression; this does not synthesize an identity or guarantee likeness.\')\nclass CommercialMeshSwap:\n @classmethod\n def INPUT_TYPES(cls):return {\'required\':{\'source\':(\'IMAGE\',),\'target\':(\'IMAGE\',),\'source_index\':(\'INT\',{\'default\':0,\'min\':0,\'max\':9}),\'target_index\':(\'INT\',{\'default\':0,\'min\':0,\'max\':9}),\'blend_mode\':([\'Feather\',\'Poisson\'],)}}\n RETURN_TYPES=(\'IMAGE\',\'STRING\');RETURN_NAMES=(\'image\',\'report\');FUNCTION=\'swap\';CATEGORY=\'Image Studio Commercial/Face\'\n def swap(self,source,target,source_index,target_index,blend_mode="Feather"):\n  out,report=composite(rgb(source),rgb(target),source_index,target_index,blend_mode)\n  return torch.from_numpy(out.astype(np.float32)/255).unsqueeze(0),json.dumps(report)\nclass CommercialMaskedEdit:\n @classmethod\n def INPUT_TYPES(cls):return {\'required\':{\'original\':(\'IMAGE\',),\'edited\':(\'IMAGE\',),\'mask\':(\'MASK\',),\'feather\':(\'INT\',{\'default\':12,\'min\':0,\'max\':128})}}\n RETURN_TYPES=(\'IMAGE\',);FUNCTION=\'paste\';CATEGORY=\'Image Studio Commercial/Edit\'\n def paste(self,original,edited,mask,feather):\n  a,b=rgb(original),rgb(edited);h,w=a.shape[:2];b=cv2.resize(b,(w,h),interpolation=cv2.INTER_LANCZOS4)\n  m=mask[0].cpu().numpy();m=cv2.resize(m,(w,h));m=np.clip(m,0,1)\n  if not m.any():raise ValueError(\'Paint a mask in LoadImage → MaskEditor first (white = replace).\')\n  if feather:m=cv2.GaussianBlur(m,(0,0),feather/3)\n  out=a*(1-m[...,None])+b*m[...,None]\n  return (torch.from_numpy(out.astype(np.float32)/255).unsqueeze(0),)\nNODE_CLASS_MAPPINGS={c.__name__:c for c in [CommercialMeshSwap,CommercialMaskedEdit]}\n'
CATALOG = {'profile': 'commercial-v7', 'reviewed': '2026-11-09', 'assets': [{'pack': 'create_edit', 'folder': 'diffusion_models', 'name': 'flux-2-klein-4b.safetensors', 'bytes': 7751105712, 'sha256': 'ec3d4e733a771f61c052fb4856c48b336c55eaf2c65487c2a1faeb9bbda7a343', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/5b3abca5c0dc1ecd4eaf3340868d09063fc1c965/split_files/diffusion_models/flux-2-klein-4b.safetensors'}, {'pack': 'create_edit', 'folder': 'text_encoders', 'name': 'qwen_3_4b.safetensors', 'bytes': 8044982048, 'sha256': '6c671498573ac2f7a5501502ccce8d2b08ea6ca2f661c458e708f36b36edfc5a', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/5b3abca5c0dc1ecd4eaf3340868d09063fc1c965/split_files/text_encoders/qwen_3_4b.safetensors'}, {'pack': 'create_edit', 'folder': 'vae', 'name': 'flux2-vae.safetensors', 'bytes': 336211292, 'sha256': '868fe7b343cc8f3a19dbcfcafbc3d5f888802be3f89bd81b65b3621a066ce8f3', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/5b3abca5c0dc1ecd4eaf3340868d09063fc1c965/split_files/vae/flux2-vae.safetensors'}, {'pack': 'finish', 'folder': 'upscale_models', 'name': 'RealESRGAN_x4plus.pth', 'bytes': 67040989, 'sha256': '4fa0d38905f75ac06eb49a7951b426670021be3018265fd191d2125df9d682f1', 'license': 'BSD-3-Clause', 'source': 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'}, {'pack': 'finish', 'folder': 'diffusion_models', 'name': 'seedvr2_3b_int8_convrot.safetensors', 'bytes': 3458259704, 'sha256': 'c3dec8bcc5916843a8a858572970597462e1f2dc598d6dfd818f6cd40f53a157', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/diffusion_models/seedvr2_3b_int8_convrot.safetensors'}, {'pack': 'finish', 'folder': 'vae', 'name': 'seedvr2_ema_vae_fp16.safetensors', 'bytes': 501324814, 'sha256': '20678548f420d98d26f11442d3528f8b8c94e57ee046ef93dbb7633da8612ca1', 'license': 'Apache-2.0', 'source': 'https://huggingface.co/Comfy-Org/SeedVR2/resolve/df48879708206a403d2a61acd55578c2e80fd233/vae/seedvr2_ema_vae_fp16.safetensors'}, {'pack': 'finish', 'folder': 'mediapipe', 'name': 'face_landmarker.task', 'bytes': 3758596, 'sha256': '64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff', 'license': 'Apache-2.0', 'source': 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'}, {'pack': 'finish', 'folder': 'loras', 'name': 'bfs_head_v1_flux-klein_4b.safetensors', 'bytes': 369644384, 'sha256': '77aabcd6144612dd4873080b99283dd9a27977712548bee161894f62c91ff09c', 'license': 'MIT', 'source': 'https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/resolve/2209ae537ac5e883d6ec7db5b7bd6dccf1a0eba7/bfs_head_v1_flux-klein_4b.safetensors'}]}
LICENSE_FILES = {'realesrgan-BSD.txt': 'BSD 3-Clause License\n\nCopyright (c) 2021, Xintao Wang\nAll rights reserved.\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n1. Redistributions of source code must retain the above copyright notice, this\n   list of conditions and the following disclaimer.\n\n2. Redistributions in binary form must reproduce the above copyright notice,\n   this list of conditions and the following disclaimer in the documentation\n   and/or other materials provided with the distribution.\n\n3. Neither the name of the copyright holder nor the names of its\n   contributors may be used to endorse or promote products derived from\n   this software without specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"\nAND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\nIMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\nFOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\nOR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\nOF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'mediapipe-model-evidence.txt': 'MediaPipe model license evidence, reviewed 2026-11-09. Bundle SHA256: 64184e229b263107bc2b804c6625db1341ff2bb731874b0bcc2fe6544e0bc9ff\n\nhttps://storage.googleapis.com/mediapipe-assets/Model%20Card%20MediaPipe%20Face%20Mesh%20V2.pdf\nGeng Yan, Google\nIvan Grishchenko, Google\nMODEL DATE\nSeptember 15, 2022\nLICENSED UNDER\nApache License, Version 2.0\n\n_\nBased on Model Cards for Model Repo ing, In Proceedings of FAT* Conference (FAT*2019). ACM, New York, NY, USA\n\nhttps://storage.googleapis.com/mediapipe-assets/MediaPipe%20BlazeFace%20Model%20Card%20(Short%20Range).pdf\n\nDOCUMENTATION\nPaper: https://arxiv.org/abs/2006.10204\n\nLICENSED UNDER\nApache License, Version 2.0\n\nBased on Model Cards for Model Reporting, In Proceedings of FAT* Conference (FAT*2019). ACM, New York, NY, USA\n\n\nhttps://storage.googleapis.com/mediapipe-assets/Model%20Card%20Blendshape%20V2.pdf\nAppendix.\n\nfront-facing camera\n\nLICENSED UNDER\nApache License, Version 2.0\nLe : Input frame with predicted facial landmarks.\nRight: 3D avatar controlled with predicted blendshape\ncoe cients\n', 'flux2-klein4b-APACHE.txt': '                             Apache License\n                       Version 2.0, January 2004\n                    http://www.apache.org/licenses/\n\nTERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n1. Definitions.\n\n  "License" shall mean the terms and conditions for use, reproduction,\n  and distribution as defined by Sections 1 through 9 of this document.\n\n  "Licensor" shall mean the copyright owner or entity authorized by\n  the copyright owner that is granting the License.\n\n  "Legal Entity" shall mean the union of the acting entity and all\n  other entities that control, are controlled by, or are under common\n  control with that entity. For the purposes of this definition,\n  "control" means (i) the power, direct or indirect, to cause the\n  direction or management of such entity, whether by contract or\n  otherwise, or (ii) ownership of fifty percent (50%) or more of the\n  outstanding shares, or (iii) beneficial ownership of such entity.\n\n  "You" (or "Your") shall mean an individual or Legal Entity\n  exercising permissions granted by this License.\n\n  "Source" form shall mean the preferred form for making modifications,\n  including but not limited to software source code, documentation\n  source, and configuration files.\n\n  "Object" form shall mean any form resulting from mechanical\n  transformation or translation of a Source form, including but\n  not limited to compiled object code, generated documentation,\n  and conversions to other media types.\n\n  "Work" shall mean the work of authorship, whether in Source or\n  Object form, made available under the License, as indicated by a\n  copyright notice that is included in or attached to the work\n  (an example is provided in the Appendix below).\n\n  "Derivative Works" shall mean any work, whether in Source or Object\n  form, that is based on (or derived from) the Work and for which the\n  editorial revisions, annotations, elaborations, or other modifications\n  represent, as a whole, an original work of authorship. For the purposes\n  of this License, Derivative Works shall not include works that remain\n  separable from, or merely link (or bind by name) to the interfaces of,\n  the Work and Derivative Works thereof.\n\n  "Contribution" shall mean any work of authorship, including\n  the original version of the Work and any modifications or additions\n  to that Work or Derivative Works thereof, that is intentionally\n  submitted to Licensor for inclusion in the Work by the copyright owner\n  or by an individual or Legal Entity authorized to submit on behalf of\n  the copyright owner. For the purposes of this definition, "submitted"\n  means any form of electronic, verbal, or written communication sent\n  to the Licensor or its representatives, including but not limited to\n  communication on electronic mailing lists, source code control systems,\n  and issue tracking systems that are managed by, or on behalf of, the\n  Licensor for the purpose of discussing and improving the Work, but\n  excluding communication that is conspicuously marked or otherwise\n  designated in writing by the copyright owner as "Not a Contribution."\n\n  "Contributor" shall mean Licensor and any individual or Legal Entity\n  on behalf of whom a Contribution has been received by Licensor and\n  subsequently incorporated within the Work.\n\n2. Grant of Copyright License. Subject to the terms and conditions of\n  this License, each Contributor hereby grants to You a perpetual,\n  worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n  copyright license to reproduce, prepare Derivative Works of,\n  publicly display, publicly perform, sublicense, and distribute the\n  Work and such Derivative Works in Source or Object form.\n\n3. Grant of Patent License. Subject to the terms and conditions of\n  this License, each Contributor hereby grants to You a perpetual,\n  worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n  (except as stated in this section) patent license to make, have made,\n  use, offer to sell, sell, import, and otherwise transfer the Work,\n  where such license applies only to those patent claims licensable\n  by such Contributor that are necessarily infringed by their\n  Contribution(s) alone or by combination of their Contribution(s)\n  with the Work to which such Contribution(s) was submitted. If You\n  institute patent litigation against any entity (including a\n  cross-claim or counterclaim in a lawsuit) alleging that the Work\n  or a Contribution incorporated within the Work constitutes direct\n  or contributory patent infringement, then any patent licenses\n  granted to You under this License for that Work shall terminate\n  as of the date such litigation is filed.\n\n4. Redistribution. You may reproduce and distribute copies of the\n  Work or Derivative Works thereof in any medium, with or without\n  modifications, and in Source or Object form, provided that You\n  meet the following conditions:\n\n  (a) You must give any other recipients of the Work or\n      Derivative Works a copy of this License; and\n\n  (b) You must cause any modified files to carry prominent notices\n      stating that You changed the files; and\n\n  (c) You must retain, in the Source form of any Derivative Works\n      that You distribute, all copyright, patent, trademark, and\n      attribution notices from the Source form of the Work,\n      excluding those notices that do not pertain to any part of\n      the Derivative Works; and\n\n  (d) If the Work includes a "NOTICE" text file as part of its\n      distribution, then any Derivative Works that You distribute must\n      include a readable copy of the attribution notices contained\n      within such NOTICE file, excluding those notices that do not\n      pertain to any part of the Derivative Works, in at least one\n      of the following places: within a NOTICE text file distributed\n      as part of the Derivative Works; within the Source form or\n      documentation, if provided along with the Derivative Works; or,\n      within a display generated by the Derivative Works, if and\n      wherever such third-party notices normally appear. The contents\n      of the NOTICE file are for informational purposes only and\n      do not modify the License. You may add Your own attribution\n      notices within Derivative Works that You distribute, alongside\n      or as an addendum to the NOTICE text from the Work, provided\n      that such additional attribution notices cannot be construed\n      as modifying the License.\n\n  You may add Your own copyright statement to Your modifications and\n  may provide additional or different license terms and conditions\n  for use, reproduction, or distribution of Your modifications, or\n  for any such Derivative Works as a whole, provided Your use,\n  reproduction, and distribution of the Work otherwise complies with\n  the conditions stated in this License.\n\n5. Submission of Contributions. Unless You explicitly state otherwise,\n  any Contribution intentionally submitted for inclusion in the Work\n  by You to the Licensor shall be under the terms and conditions of\n  this License, without any additional terms or conditions.\n  Notwithstanding the above, nothing herein shall supersede or modify\n  the terms of any separate license agreement you may have executed\n  with Licensor regarding such Contributions.\n\n6. Trademarks. This License does not grant permission to use the trade\n  names, trademarks, service marks, or product names of the Licensor,\n  except as required for reasonable and customary use in describing the\n  origin of the Work and reproducing the content of the NOTICE file.\n\n7. Disclaimer of Warranty. Unless required by applicable law or\n  agreed to in writing, Licensor provides the Work (and each\n  Contributor provides its Contributions) on an "AS IS" BASIS,\n  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n  implied, including, without limitation, any warranties or conditions\n  of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n  PARTICULAR PURPOSE. You are solely responsible for determining the\n  appropriateness of using or redistributing the Work and assume any\n  risks associated with Your exercise of permissions under this License.\n\n8. Limitation of Liability. In no event and under no legal theory,\n  whether in tort (including negligence), contract, or otherwise,\n  unless required by applicable law (such as deliberate and grossly\n  negligent acts) or agreed to in writing, shall any Contributor be\n  liable to You for damages, including any direct, indirect, special,\n  incidental, or consequential damages of any character arising as a\n  result of this License or out of the use or inability to use the\n  Work (including but not limited to damages for loss of goodwill,\n  work stoppage, computer failure or malfunction, or any and all\n  other commercial damages or losses), even if such Contributor\n  has been advised of the possibility of such damages.\n\n9. Accepting Warranty or Additional Liability. While redistributing\n  the Work or Derivative Works thereof, You may choose to offer,\n  and charge a fee for, acceptance of support, warranty, indemnity,\n  or other liability obligations and/or rights consistent with this\n  License. However, in accepting such obligations, You may act only\n  on Your own behalf and on Your sole responsibility, not on behalf\n  of any other Contributor, and only if You agree to indemnify,\n  defend, and hold each Contributor harmless for any liability\n  incurred by, or claims asserted against, such Contributor by reason\n  of your accepting any such warranty or additional liability.\n\nEND OF TERMS AND CONDITIONS\n', 'bfs-model-card-MIT.md': 'Original model card snapshot, declared MIT. Source: https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/blob/2209ae537ac5e883d6ec7db5b7bd6dccf1a0eba7/README.md\n\n---\nlicense: mit\nbase_model:\n- Qwen/Qwen-Image-2.1\n- Qwen/Qwen-Image-Edit-2511\n- black-forest-labs/FLUX.2-klein-9B\n- black-forest-labs/FLUX.2-klein-4B\n- black-forest-labs/FLUX.2-klein-base-4B\n- black-forest-labs/FLUX.2-klein-base-9B\n- black-forest-labs/FLUX.2-klein-9b-fp8\n- black-forest-labs/FLUX.2-klein-4b-fp8\n- krea/Krea-2-Raw\npipeline_tag: image-to-image\ntags:\n- lora\n- qwen-image\n- qwen-image-2.1\n- qwen-image-edit\n- face-swap\n- head-swap\n- body-swap\n- selective-body-swap\n- image-editing\n- comfyui\n- bfs\nlanguage:\n- en\nlibrary_name: diffusers\n---\n\n# BFS: Best Face Swap\n\nBFS is a collection of LoRAs for face, head, and body replacement across Qwen Image 2.1, Qwen\nImage Edit 2509/2511, Flux 2 Klein, Krea 2, and experimental LTX-2 workflows. Each release has\nits own base-model, input-order, prompt, and workflow contract. Use the guide for the exact\nversion you are loading.\n\n## Guides\n\n- [BFS Head V1 / V1.1 for Qwen Image 2.1](docs/qwen-image-2.1.md)\n- [BFS Body Swap V1.0 for Qwen Image 2.1](docs/qwen-image-2.1-body.md)\n- [BFS Body Swap Selective (Magenta) V1.0 for Qwen Image 2.1](docs/qwen-image-2.1-body-selective.md):\n  replaces only the person marked in magenta, for group photos and busy scenes\n- [Qwen Image Edit 2509/2511](docs/qwen-image-edit.md)\n- [Flux 2 Klein](docs/flux-2-klein.md)\n- [Krea 2](docs/krea-2.md)\n- [Experimental LTX-2 weights](docs/ltx-2.md)\n\n## Downloads at a glance\n\n| Family | Weights | Workflow |\n|---|---|---|\n| Qwen Image 2.1 (head) | [`bfs_head_v1.1_qwen_2.1.safetensors`](bfs_head_v1.1_qwen_2.1.safetensors) (recommended), [`bfs_head_v1.1_alternative_qwen_2.1.safetensors`](bfs_head_v1.1_alternative_qwen_2.1.safetensors), [`bfs_head_v1_qwen_2.1.safetensors`](bfs_head_v1_qwen_2.1.safetensors) | [`Head Swap V1 Qwen 2.1 Workflow.json`](workflows/Head%20Swap%20V1%20Qwen%202.1%20Workflow.json) |\n| Qwen Image 2.1 (body) | [`bfs_body_swap_v1.0_qwen_2.1.safetensors`](bfs_body_swap_v1.0_qwen_2.1.safetensors) | [`Body Swap Qwen Image 2.1 - V1.json`](workflows/Body%20Swap%20Qwen%20Image%202.1%20-%20V1.json), see the [body swap guide](docs/qwen-image-2.1-body.md) |\n| Qwen Image 2.1 (selective body, magenta mark) | [`bfs_body_swap_selective_magenta_v1.0_qwen_2.1.safetensors`](bfs_body_swap_selective_magenta_v1.0_qwen_2.1.safetensors) | [`Body Swap Selective Qwen Image 2.1 - V1.0.json`](workflows/Body%20Swap%20Selective%20Qwen%20Image%202.1%20-%20V1.0.json), see the [selective guide](docs/qwen-image-2.1-body-selective.md) |\n| Qwen Image Edit 2509 | [`bfs_face_v1_qwen_image_edit_2509.safetensors`](bfs_face_v1_qwen_image_edit_2509.safetensors), [`bfs_head_v1_qwen_image_edit_2509.safetensors`](bfs_head_v1_qwen_image_edit_2509.safetensors), [`bfs_head_v2_qwen_image_edit_2509.safetensors`](bfs_head_v2_qwen_image_edit_2509.safetensors), [`bfs_head_v3_qwen_image_edit_2509.safetensors`](bfs_head_v3_qwen_image_edit_2509.safetensors), [`bfs_head_v4_qwen_image_edit_2509.safetensors`](bfs_head_v4_qwen_image_edit_2509.safetensors) | See the [Qwen Edit guide](docs/qwen-image-edit.md) |\n| Qwen Image Edit 2511 | [`bfs_head_v5_2511_original.safetensors`](bfs_head_v5_2511_original.safetensors), [`bfs_head_v5_2511_merged_version_rank_16_fp16.safetensors`](bfs_head_v5_2511_merged_version_rank_16_fp16.safetensors), [`bfs_head_v5_2511_merged_version_rank_32_fp32.safetensors`](bfs_head_v5_2511_merged_version_rank_32_fp32.safetensors) | See the [Qwen Edit guide](docs/qwen-image-edit.md) |\n| Flux 2 Klein | [`bfs_head_v1_flux-klein_4b.safetensors`](bfs_head_v1_flux-klein_4b.safetensors), [`bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors`](bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors), [`bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors`](bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors), [`bfs_head_v1.1_optional_flux-klein_4b.safetensors`](bfs_head_v1.1_optional_flux-klein_4b.safetensors) | See the [Flux guide](docs/flux-2-klein.md) |\n| Krea 2 | [`bfs_head_swap_v1.1_krea2.safetensors`](bfs_head_swap_v1.1_krea2.safetensors), [`bfs_body_swap_v1_krea2.safetensors`](bfs_body_swap_v1_krea2.safetensors) | See the [Krea 2 guide](docs/krea-2.md) |\n| Experimental LTX-2 | [`bfs_head_swap_v1_ltx2_ic_lora_5000.safetensors`](bfs_head_swap_v1_ltx2_ic_lora_5000.safetensors), [`bfs_head_swap_v1_ltx2_ic_lora_6000.safetensors`](bfs_head_swap_v1_ltx2_ic_lora_6000.safetensors), [`bfs_head_swap_v1_ltx2_ic_lora_12000_train_3.safetensors`](bfs_head_swap_v1_ltx2_ic_lora_12000_train_3.safetensors) | See the [LTX-2 guide](docs/ltx-2.md) |\n\n## General rules\n\n- Use the LoRA with its matching base model whenever possible.\n- Follow the input order in the version guide; several BFS head releases intentionally use body\n  first and reference face second.\n- Start at LoRA strength `1.0`, then adjust for the base model, resolution, and reference image.\n- The examples and workflows in this repository are demonstrations, not a guarantee of identical\n  results on every image.\n- Do not use or share results involving public figures or people who have not given consent.\n\n## Examples\n\nThe existing examples are grouped by workflow family in the individual guides. Qwen Image 2.1\nexamples are `images/bfs_qwen_2.1_1.png` through `images/bfs_qwen_2.1_8.png`.\n\n## Support\n\nFeel free to experiment and share your results responsibly. For workflow issues, first update\nComfyUI and verify that the base model, text encoder, VAE, workflow, and LoRA all belong to the\nsame model family.\n', 'seedvr2-APACHE.txt': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "{}"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2025 seed\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n', 'qwen3-4b-APACHE.txt': '\n                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright 2024 Alibaba Cloud\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.', 'bfs-klein-guide.md': 'Source: https://huggingface.co/Alissonerdx/BFS-Best-Face-Swap/blob/2209ae537ac5e883d6ec7db5b7bd6dccf1a0eba7/docs/flux-2-klein.md\n\n# Flux 2 Klein Head-Swap LoRAs\n\nThese BFS weights target Flux 2 Klein 4B/9B variants. They use the body-first input contract:\n\n- **Image 1:** target body/base image\n- **Image 2:** reference face/head\n\nDo not reverse the images for these releases.\n\n## Downloads\n\n- [`bfs_head_v1_flux-klein_4b.safetensors`](../bfs_head_v1_flux-klein_4b.safetensors)\n- [`bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors`](../bfs_head_v1_flux-klein_9b_step3500_rank128.safetensors)\n- [`bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors`](../bfs_head_v1_flux-klein_9b_step3750_rank64.safetensors)\n- [`bfs_head_v1.1_optional_flux-klein_4b.safetensors`](../bfs_head_v1.1_optional_flux-klein_4b.safetensors)\n\n## Prompt\n\n```text\nhead_swap: start with Picture 1 as the base image, keeping its lighting, environment, and background. remove the head from Picture 1 completely and replace it with the head from Picture 2, strictly preserving the hair, eye color, and nose structure of Picture 2. copy the direction of the eye, head rotation, and micro-expressions from Picture 1. Keep the body, clothing, pose, framing, and background intact.\n```\n\nYou can add a short expression or pose instruction when it is visible in the target image. Keep\nthe edit focused on the head and avoid requesting an unrelated scene change in the same prompt.\n\n## Workflow\n\n[`workflows/Head Swap V1 Flux 2 Klein 4b_9b (base_distill).json`](../workflows/Head%20Swap%20V1%20Flux%202%20Klein%204b_9b%20(base_distill).json)\n\nStart around LoRA strength `1.0`; adjust after checking identity preservation, hair boundaries,\nexpression transfer, and skin blending.\n\n[Back to the repository index](../README.md)\n', 'mediapipe-source-APACHE.txt': '                                 Apache License\n                           Version 2.0, January 2004\n                        http://www.apache.org/licenses/\n\n   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n\n   1. Definitions.\n\n      "License" shall mean the terms and conditions for use, reproduction,\n      and distribution as defined by Sections 1 through 9 of this document.\n\n      "Licensor" shall mean the copyright owner or entity authorized by\n      the copyright owner that is granting the License.\n\n      "Legal Entity" shall mean the union of the acting entity and all\n      other entities that control, are controlled by, or are under common\n      control with that entity. For the purposes of this definition,\n      "control" means (i) the power, direct or indirect, to cause the\n      direction or management of such entity, whether by contract or\n      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n      outstanding shares, or (iii) beneficial ownership of such entity.\n\n      "You" (or "Your") shall mean an individual or Legal Entity\n      exercising permissions granted by this License.\n\n      "Source" form shall mean the preferred form for making modifications,\n      including but not limited to software source code, documentation\n      source, and configuration files.\n\n      "Object" form shall mean any form resulting from mechanical\n      transformation or translation of a Source form, including but\n      not limited to compiled object code, generated documentation,\n      and conversions to other media types.\n\n      "Work" shall mean the work of authorship, whether in Source or\n      Object form, made available under the License, as indicated by a\n      copyright notice that is included in or attached to the work\n      (an example is provided in the Appendix below).\n\n      "Derivative Works" shall mean any work, whether in Source or Object\n      form, that is based on (or derived from) the Work and for which the\n      editorial revisions, annotations, elaborations, or other modifications\n      represent, as a whole, an original work of authorship. For the purposes\n      of this License, Derivative Works shall not include works that remain\n      separable from, or merely link (or bind by name) to the interfaces of,\n      the Work and Derivative Works thereof.\n\n      "Contribution" shall mean any work of authorship, including\n      the original version of the Work and any modifications or additions\n      to that Work or Derivative Works thereof, that is intentionally\n      submitted to Licensor for inclusion in the Work by the copyright owner\n      or by an individual or Legal Entity authorized to submit on behalf of\n      the copyright owner. For the purposes of this definition, "submitted"\n      means any form of electronic, verbal, or written communication sent\n      to the Licensor or its representatives, including but not limited to\n      communication on electronic mailing lists, source code control systems,\n      and issue tracking systems that are managed by, or on behalf of, the\n      Licensor for the purpose of discussing and improving the Work, but\n      excluding communication that is conspicuously marked or otherwise\n      designated in writing by the copyright owner as "Not a Contribution."\n\n      "Contributor" shall mean Licensor and any individual or Legal Entity\n      on behalf of whom a Contribution has been received by Licensor and\n      subsequently incorporated within the Work.\n\n   2. Grant of Copyright License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      copyright license to reproduce, prepare Derivative Works of,\n      publicly display, publicly perform, sublicense, and distribute the\n      Work and such Derivative Works in Source or Object form.\n\n   3. Grant of Patent License. Subject to the terms and conditions of\n      this License, each Contributor hereby grants to You a perpetual,\n      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n      (except as stated in this section) patent license to make, have made,\n      use, offer to sell, sell, import, and otherwise transfer the Work,\n      where such license applies only to those patent claims licensable\n      by such Contributor that are necessarily infringed by their\n      Contribution(s) alone or by combination of their Contribution(s)\n      with the Work to which such Contribution(s) was submitted. If You\n      institute patent litigation against any entity (including a\n      cross-claim or counterclaim in a lawsuit) alleging that the Work\n      or a Contribution incorporated within the Work constitutes direct\n      or contributory patent infringement, then any patent licenses\n      granted to You under this License for that Work shall terminate\n      as of the date such litigation is filed.\n\n   4. Redistribution. You may reproduce and distribute copies of the\n      Work or Derivative Works thereof in any medium, with or without\n      modifications, and in Source or Object form, provided that You\n      meet the following conditions:\n\n      (a) You must give any other recipients of the Work or\n          Derivative Works a copy of this License; and\n\n      (b) You must cause any modified files to carry prominent notices\n          stating that You changed the files; and\n\n      (c) You must retain, in the Source form of any Derivative Works\n          that You distribute, all copyright, patent, trademark, and\n          attribution notices from the Source form of the Work,\n          excluding those notices that do not pertain to any part of\n          the Derivative Works; and\n\n      (d) If the Work includes a "NOTICE" text file as part of its\n          distribution, then any Derivative Works that You distribute must\n          include a readable copy of the attribution notices contained\n          within such NOTICE file, excluding those notices that do not\n          pertain to any part of the Derivative Works, in at least one\n          of the following places: within a NOTICE text file distributed\n          as part of the Derivative Works; within the Source form or\n          documentation, if provided along with the Derivative Works; or,\n          within a display generated by the Derivative Works, if and\n          wherever such third-party notices normally appear. The contents\n          of the NOTICE file are for informational purposes only and\n          do not modify the License. You may add Your own attribution\n          notices within Derivative Works that You distribute, alongside\n          or as an addendum to the NOTICE text from the Work, provided\n          that such additional attribution notices cannot be construed\n          as modifying the License.\n\n      You may add Your own copyright statement to Your modifications and\n      may provide additional or different license terms and conditions\n      for use, reproduction, or distribution of Your modifications, or\n      for any such Derivative Works as a whole, provided Your use,\n      reproduction, and distribution of the Work otherwise complies with\n      the conditions stated in this License.\n\n   5. Submission of Contributions. Unless You explicitly state otherwise,\n      any Contribution intentionally submitted for inclusion in the Work\n      by You to the Licensor shall be under the terms and conditions of\n      this License, without any additional terms or conditions.\n      Notwithstanding the above, nothing herein shall supersede or modify\n      the terms of any separate license agreement you may have executed\n      with Licensor regarding such Contributions.\n\n   6. Trademarks. This License does not grant permission to use the trade\n      names, trademarks, service marks, or product names of the Licensor,\n      except as required for reasonable and customary use in describing the\n      origin of the Work and reproducing the content of the NOTICE file.\n\n   7. Disclaimer of Warranty. Unless required by applicable law or\n      agreed to in writing, Licensor provides the Work (and each\n      Contributor provides its Contributions) on an "AS IS" BASIS,\n      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n      implied, including, without limitation, any warranties or conditions\n      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n      PARTICULAR PURPOSE. You are solely responsible for determining the\n      appropriateness of using or redistributing the Work and assume any\n      risks associated with Your exercise of permissions under this License.\n\n   8. Limitation of Liability. In no event and under no legal theory,\n      whether in tort (including negligence), contract, or otherwise,\n      unless required by applicable law (such as deliberate and grossly\n      negligent acts) or agreed to in writing, shall any Contributor be\n      liable to You for damages, including any direct, indirect, special,\n      incidental, or consequential damages of any character arising as a\n      result of this License or out of the use or inability to use the\n      Work (including but not limited to damages for loss of goodwill,\n      work stoppage, computer failure or malfunction, or any and all\n      other commercial damages or losses), even if such Contributor\n      has been advised of the possibility of such damages.\n\n   9. Accepting Warranty or Additional Liability. While redistributing\n      the Work or Derivative Works thereof, You may choose to offer,\n      and charge a fee for, acceptance of support, warranty, indemnity,\n      or other liability obligations and/or rights consistent with this\n      License. However, in accepting such obligations, You may act only\n      on Your own behalf and on Your sole responsibility, not on behalf\n      of any other Contributor, and only if You agree to indemnify,\n      defend, and hold each Contributor harmless for any liability\n      incurred by, or claims asserted against, such Contributor by reason\n      of your accepting any such warranty or additional liability.\n\n   END OF TERMS AND CONDITIONS\n\n   APPENDIX: How to apply the Apache License to your work.\n\n      To apply the Apache License to your work, attach the following\n      boilerplate notice, with the fields enclosed by brackets "[]"\n      replaced with your own identifying information. (Don\'t include\n      the brackets!)  The text should be enclosed in the appropriate\n      comment syntax for the file format. We also recommend that a\n      file or class name and description of purpose be included on the\n      same "printed page" as the copyright notice for easier\n      identification within third-party archives.\n\n   Copyright [yyyy] [name of copyright owner]\n\n   Licensed under the Apache License, Version 2.0 (the "License");\n   you may not use this file except in compliance with the License.\n   You may obtain a copy of the License at\n\n       http://www.apache.org/licenses/LICENSE-2.0\n\n   Unless required by applicable law or agreed to in writing, software\n   distributed under the License is distributed on an "AS IS" BASIS,\n   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n   See the License for the specific language governing permissions and\n   limitations under the License.\n\n===========================================================================\nFor files under tasks/cc/text/language_detector/custom_ops/utils/utf/\n===========================================================================\n/*\n * The authors of this software are Rob Pike and Ken Thompson.\n *              Copyright (c) 2002 by Lucent Technologies.\n * Permission to use, copy, modify, and distribute this software for any\n * purpose without fee is hereby granted, provided that this entire notice\n * is included in all copies of any software which is or includes a copy\n * or modification of this software and in all copies of the supporting\n * documentation for such software.\n * THIS SOFTWARE IS BEING PROVIDED "AS IS", WITHOUT ANY EXPRESS OR IMPLIED\n * WARRANTY.  IN PARTICULAR, NEITHER THE AUTHORS NOR LUCENT TECHNOLOGIES MAKE ANY\n * REPRESENTATION OR WARRANTY OF ANY KIND CONCERNING THE MERCHANTABILITY\n * OF THIS SOFTWARE OR ITS FITNESS FOR ANY PARTICULAR PURPOSE.\n */\n'}
WORKFLOW_PAYLOAD = {'Create - Cinematic': {'last_node_id': 15, 'last_link_id': 17, 'nodes': [{'id': 1, 'type': 'UNETLoader', 'pos': [0, 0], 'size': [330, 270], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [7]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 2, 'type': 'CLIPLoader', 'pos': [0, 320], 'size': [330, 270], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 3, 'type': 'VAELoader', 'pos': [0, 640], 'size': [330, 270], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [16]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 4, 'type': 'CLIPTextEncode', 'pos': [370, 0], 'size': [330, 270], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [2, 8]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Cinematic photograph of a rain-soaked city street at blue hour, a lone traveller beneath warm shop lights, grounded realistic materials, subtle film grain, natural skin texture, thoughtful framing, dramatic but believable lighting. Describe your own scene here.'], 'title': 'PROMPT — describe your image or edit'}, {'id': 5, 'type': 'ConditioningZeroOut', 'pos': [370, 320], 'size': [330, 270], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [9]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 6, 'type': 'PrimitiveInt', 'pos': [370, 640], 'size': [330, 270], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'INT', 'type': 'INT', 'links': [3, 4]}], 'properties': {'Node name for S&R': 'PrimitiveInt'}, 'widgets_values': [1216, 'fixed'], 'title': 'WIDTH — multiples of 16'}, {'id': 7, 'type': 'PrimitiveInt', 'pos': [740, 0], 'size': [330, 270], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'INT', 'type': 'INT', 'links': [5, 6]}], 'properties': {'Node name for S&R': 'PrimitiveInt'}, 'widgets_values': [832, 'fixed'], 'title': 'HEIGHT — multiples of 16'}, {'id': 8, 'type': 'EmptyFlux2LatentImage', 'pos': [740, 320], 'size': [330, 270], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 3}, {'name': 'height', 'type': 'INT', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [14]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 9, 'type': 'Flux2Scheduler', 'pos': [740, 640], 'size': [330, 270], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 4}, {'name': 'height', 'type': 'INT', 'link': 6}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [13]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 10, 'type': 'RandomNoise', 'pos': [1110, 0], 'size': [330, 270], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [10]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'SEED'}, {'id': 11, 'type': 'KSamplerSelect', 'pos': [1110, 320], 'size': [330, 270], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [12]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 12, 'type': 'CFGGuider', 'pos': [1110, 640], 'size': [330, 270], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 7}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 8}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 9}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [11]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 13, 'type': 'SamplerCustomAdvanced', 'pos': [1480, 0], 'size': [330, 270], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 10}, {'name': 'guider', 'type': 'GUIDER', 'link': 11}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 12}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 13}, {'name': 'latent_image', 'type': 'LATENT', 'link': 14}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [15]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 14, 'type': 'VAEDecodeTiled', 'pos': [1480, 320], 'size': [330, 270], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 15}, {'name': 'vae', 'type': 'VAE', 'link': 16}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [17]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 15, 'type': 'SaveImage', 'pos': [1480, 640], 'size': [330, 270], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 17}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Create'], 'title': 'SAVE IMAGE — original image, no lettering'}], 'links': [[1, 2, 0, 4, 0, 'CLIP'], [2, 4, 0, 5, 0, 'CONDITIONING'], [3, 6, 0, 8, 0, 'INT'], [4, 6, 0, 9, 0, 'INT'], [5, 7, 0, 8, 1, 'INT'], [6, 7, 0, 9, 1, 'INT'], [7, 1, 0, 12, 0, 'MODEL'], [8, 4, 0, 12, 1, 'CONDITIONING'], [9, 5, 0, 12, 2, 'CONDITIONING'], [10, 10, 0, 13, 0, 'NOISE'], [11, 12, 0, 13, 1, 'GUIDER'], [12, 11, 0, 13, 2, 'SAMPLER'], [13, 9, 0, 13, 3, 'SIGMAS'], [14, 8, 0, 13, 4, 'LATENT'], [15, 13, 0, 14, 0, 'LATENT'], [16, 3, 0, 14, 1, 'VAE'], [17, 14, 0, 15, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.75, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Image Edit': {'last_node_id': 19, 'last_link_id': 25, 'nodes': [{'id': 1, 'type': 'UNETLoader', 'pos': [0, 0], 'size': [330, 270], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [15]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 2, 'type': 'CLIPLoader', 'pos': [0, 320], 'size': [330, 270], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 3, 'type': 'VAELoader', 'pos': [0, 640], 'size': [330, 270], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [5, 24]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 4, 'type': 'CLIPTextEncode', 'pos': [370, 0], 'size': [330, 270], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [2, 6]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Edit image 1: describe the change you want here. Preserve the original composition and everything unrelated to the requested change.'], 'title': 'PROMPT — describe your image or edit'}, {'id': 5, 'type': 'ConditioningZeroOut', 'pos': [370, 320], 'size': [330, 270], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [8]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 6, 'type': 'LoadImage', 'pos': [370, 640], 'size': [330, 270], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'TARGET — image to edit'}, {'id': 7, 'type': 'ImageScaleToTotalPixels', 'pos': [740, 0], 'size': [330, 270], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 3}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4, 10]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 16], 'title': 'Reference size — about 1 megapixel'}, {'id': 8, 'type': 'VAEEncode', 'pos': [740, 320], 'size': [330, 270], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 4}, {'name': 'vae', 'type': 'VAE', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [7, 9]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 9, 'type': 'ReferenceLatent', 'pos': [740, 640], 'size': [330, 270], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 6}, {'name': 'latent', 'type': 'LATENT', 'link': 7}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [16]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 10, 'type': 'ReferenceLatent', 'pos': [1110, 0], 'size': [330, 270], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 8}, {'name': 'latent', 'type': 'LATENT', 'link': 9}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [17]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 11, 'type': 'GetImageSize', 'pos': [1110, 320], 'size': [330, 270], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 10}], 'outputs': [{'name': 'width', 'type': 'INT', 'links': [11, 12]}, {'name': 'height', 'type': 'INT', 'links': [13, 14]}, {'name': 'batch_size', 'type': 'INT', 'links': []}], 'properties': {'Node name for S&R': 'GetImageSize'}, 'widgets_values': [], 'title': 'Use target image aspect ratio'}, {'id': 12, 'type': 'EmptyFlux2LatentImage', 'pos': [1110, 640], 'size': [330, 270], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 11}, {'name': 'height', 'type': 'INT', 'link': 13}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [22]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 13, 'type': 'Flux2Scheduler', 'pos': [1480, 0], 'size': [330, 270], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 12}, {'name': 'height', 'type': 'INT', 'link': 14}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [21]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 14, 'type': 'RandomNoise', 'pos': [1480, 320], 'size': [330, 270], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [18]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'SEED'}, {'id': 15, 'type': 'KSamplerSelect', 'pos': [1480, 640], 'size': [330, 270], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [20]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 16, 'type': 'CFGGuider', 'pos': [1850, 0], 'size': [330, 270], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 15}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 16}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 17}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [19]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 17, 'type': 'SamplerCustomAdvanced', 'pos': [1850, 320], 'size': [330, 270], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 18}, {'name': 'guider', 'type': 'GUIDER', 'link': 19}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 20}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 21}, {'name': 'latent_image', 'type': 'LATENT', 'link': 22}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [23]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 18, 'type': 'VAEDecodeTiled', 'pos': [1850, 640], 'size': [330, 270], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 23}, {'name': 'vae', 'type': 'VAE', 'link': 24}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [25]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 19, 'type': 'SaveImage', 'pos': [2220, 0], 'size': [330, 270], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 25}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/ImageEdit'], 'title': 'SAVE IMAGE — original image, no lettering'}], 'links': [[1, 2, 0, 4, 0, 'CLIP'], [2, 4, 0, 5, 0, 'CONDITIONING'], [3, 6, 0, 7, 0, 'IMAGE'], [4, 7, 0, 8, 0, 'IMAGE'], [5, 3, 0, 8, 1, 'VAE'], [6, 4, 0, 9, 0, 'CONDITIONING'], [7, 8, 0, 9, 1, 'LATENT'], [8, 5, 0, 10, 0, 'CONDITIONING'], [9, 8, 0, 10, 1, 'LATENT'], [10, 7, 0, 11, 0, 'IMAGE'], [11, 11, 0, 12, 0, 'INT'], [12, 11, 0, 13, 0, 'INT'], [13, 11, 1, 12, 1, 'INT'], [14, 11, 1, 13, 1, 'INT'], [15, 1, 0, 16, 0, 'MODEL'], [16, 9, 0, 16, 1, 'CONDITIONING'], [17, 10, 0, 16, 2, 'CONDITIONING'], [18, 14, 0, 17, 0, 'NOISE'], [19, 16, 0, 17, 1, 'GUIDER'], [20, 15, 0, 17, 2, 'SAMPLER'], [21, 13, 0, 17, 3, 'SIGMAS'], [22, 12, 0, 17, 4, 'LATENT'], [23, 17, 0, 18, 0, 'LATENT'], [24, 3, 0, 18, 1, 'VAE'], [25, 18, 0, 19, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.75, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Inpaint - Masked Edit': {'last_node_id': 20, 'last_link_id': 28, 'nodes': [{'id': 1, 'type': 'UNETLoader', 'pos': [0, 0], 'size': [330, 270], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [15]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 2, 'type': 'CLIPLoader', 'pos': [0, 320], 'size': [330, 270], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 3, 'type': 'VAELoader', 'pos': [0, 640], 'size': [330, 270], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [5, 24]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 4, 'type': 'CLIPTextEncode', 'pos': [370, 0], 'size': [330, 270], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [2, 6]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Edit only the selected region of image 1: describe the replacement here. Match the original lighting and perspective.'], 'title': 'PROMPT — describe your image or edit'}, {'id': 5, 'type': 'ConditioningZeroOut', 'pos': [370, 320], 'size': [330, 270], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [8]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 6, 'type': 'LoadImage', 'pos': [370, 640], 'size': [330, 270], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3, 25]}, {'name': 'MASK', 'type': 'MASK', 'links': [27]}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'TARGET — image to edit / right-click → MaskEditor'}, {'id': 7, 'type': 'ImageScaleToTotalPixels', 'pos': [740, 0], 'size': [330, 270], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 3}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4, 10]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 16], 'title': 'Reference size — about 1 megapixel'}, {'id': 8, 'type': 'VAEEncode', 'pos': [740, 320], 'size': [330, 270], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 4}, {'name': 'vae', 'type': 'VAE', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [7, 9]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 9, 'type': 'ReferenceLatent', 'pos': [740, 640], 'size': [330, 270], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 6}, {'name': 'latent', 'type': 'LATENT', 'link': 7}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [16]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 10, 'type': 'ReferenceLatent', 'pos': [1110, 0], 'size': [330, 270], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 8}, {'name': 'latent', 'type': 'LATENT', 'link': 9}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [17]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 11, 'type': 'GetImageSize', 'pos': [1110, 320], 'size': [330, 270], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 10}], 'outputs': [{'name': 'width', 'type': 'INT', 'links': [11, 12]}, {'name': 'height', 'type': 'INT', 'links': [13, 14]}, {'name': 'batch_size', 'type': 'INT', 'links': []}], 'properties': {'Node name for S&R': 'GetImageSize'}, 'widgets_values': [], 'title': 'Use target image aspect ratio'}, {'id': 12, 'type': 'EmptyFlux2LatentImage', 'pos': [1110, 640], 'size': [330, 270], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 11}, {'name': 'height', 'type': 'INT', 'link': 13}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [22]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 13, 'type': 'Flux2Scheduler', 'pos': [1480, 0], 'size': [330, 270], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 12}, {'name': 'height', 'type': 'INT', 'link': 14}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [21]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 14, 'type': 'RandomNoise', 'pos': [1480, 320], 'size': [330, 270], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [18]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'SEED'}, {'id': 15, 'type': 'KSamplerSelect', 'pos': [1480, 640], 'size': [330, 270], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [20]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 16, 'type': 'CFGGuider', 'pos': [1850, 0], 'size': [330, 270], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 15}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 16}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 17}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [19]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 17, 'type': 'SamplerCustomAdvanced', 'pos': [1850, 320], 'size': [330, 270], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 18}, {'name': 'guider', 'type': 'GUIDER', 'link': 19}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 20}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 21}, {'name': 'latent_image', 'type': 'LATENT', 'link': 22}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [23]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 18, 'type': 'VAEDecodeTiled', 'pos': [1850, 640], 'size': [330, 270], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 23}, {'name': 'vae', 'type': 'VAE', 'link': 24}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [26]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 19, 'type': 'CommercialMaskedEdit', 'pos': [2220, 0], 'size': [330, 270], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [{'name': 'original', 'type': 'IMAGE', 'link': 25}, {'name': 'edited', 'type': 'IMAGE', 'link': 26}, {'name': 'mask', 'type': 'MASK', 'link': 27}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [28]}], 'properties': {'Node name for S&R': 'CommercialMaskedEdit'}, 'widgets_values': [12], 'title': 'Paste only inside painted mask'}, {'id': 20, 'type': 'SaveImage', 'pos': [2220, 320], 'size': [330, 270], 'flags': {}, 'order': 19, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 28}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Inpaint'], 'title': 'SAVE IMAGE — original image, no lettering'}], 'links': [[1, 2, 0, 4, 0, 'CLIP'], [2, 4, 0, 5, 0, 'CONDITIONING'], [3, 6, 0, 7, 0, 'IMAGE'], [4, 7, 0, 8, 0, 'IMAGE'], [5, 3, 0, 8, 1, 'VAE'], [6, 4, 0, 9, 0, 'CONDITIONING'], [7, 8, 0, 9, 1, 'LATENT'], [8, 5, 0, 10, 0, 'CONDITIONING'], [9, 8, 0, 10, 1, 'LATENT'], [10, 7, 0, 11, 0, 'IMAGE'], [11, 11, 0, 12, 0, 'INT'], [12, 11, 0, 13, 0, 'INT'], [13, 11, 1, 12, 1, 'INT'], [14, 11, 1, 13, 1, 'INT'], [15, 1, 0, 16, 0, 'MODEL'], [16, 9, 0, 16, 1, 'CONDITIONING'], [17, 10, 0, 16, 2, 'CONDITIONING'], [18, 14, 0, 17, 0, 'NOISE'], [19, 16, 0, 17, 1, 'GUIDER'], [20, 15, 0, 17, 2, 'SAMPLER'], [21, 13, 0, 17, 3, 'SIGMAS'], [22, 12, 0, 17, 4, 'LATENT'], [23, 17, 0, 18, 0, 'LATENT'], [24, 3, 0, 18, 1, 'VAE'], [25, 6, 0, 19, 0, 'IMAGE'], [26, 18, 0, 19, 1, 'IMAGE'], [27, 6, 1, 19, 2, 'MASK'], [28, 19, 0, 20, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.75, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Face Swap - Reference Edit': {'last_node_id': 24, 'last_link_id': 32, 'nodes': [{'id': 1, 'type': 'UNETLoader', 'pos': [0, 0], 'size': [330, 270], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [22]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 2, 'type': 'CLIPLoader', 'pos': [0, 320], 'size': [330, 270], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 3, 'type': 'VAELoader', 'pos': [0, 640], 'size': [330, 270], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [5, 12, 31]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 4, 'type': 'CLIPTextEncode', 'pos': [370, 0], 'size': [330, 270], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [2, 6]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Use image 1 as the target photograph. Replace the selected person’s face identity with the person in image 2. Preserve the target pose, expression, clothing, framing and background.'], 'title': 'PROMPT — describe your image or edit'}, {'id': 5, 'type': 'ConditioningZeroOut', 'pos': [370, 320], 'size': [330, 270], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [8]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 6, 'type': 'LoadImage', 'pos': [370, 640], 'size': [330, 270], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'TARGET — image to edit'}, {'id': 7, 'type': 'LoadImage', 'pos': [740, 0], 'size': [330, 270], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [10]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'SOURCE — face / identity reference'}, {'id': 8, 'type': 'ImageScaleToTotalPixels', 'pos': [740, 320], 'size': [330, 270], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 3}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4, 17]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 16], 'title': 'Reference size — about 1 megapixel'}, {'id': 9, 'type': 'VAEEncode', 'pos': [740, 640], 'size': [330, 270], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 4}, {'name': 'vae', 'type': 'VAE', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [7, 9]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 10, 'type': 'ReferenceLatent', 'pos': [1110, 0], 'size': [330, 270], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 6}, {'name': 'latent', 'type': 'LATENT', 'link': 7}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [13]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 11, 'type': 'ReferenceLatent', 'pos': [1110, 320], 'size': [330, 270], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 8}, {'name': 'latent', 'type': 'LATENT', 'link': 9}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [15]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 12, 'type': 'ImageScaleToTotalPixels', 'pos': [1110, 640], 'size': [330, 270], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 10}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [11]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 16], 'title': 'Reference size — about 1 megapixel'}, {'id': 13, 'type': 'VAEEncode', 'pos': [1480, 0], 'size': [330, 270], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 11}, {'name': 'vae', 'type': 'VAE', 'link': 12}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [14, 16]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 14, 'type': 'ReferenceLatent', 'pos': [1480, 320], 'size': [330, 270], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 13}, {'name': 'latent', 'type': 'LATENT', 'link': 14}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [23]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 15, 'type': 'ReferenceLatent', 'pos': [1480, 640], 'size': [330, 270], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 15}, {'name': 'latent', 'type': 'LATENT', 'link': 16}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [24]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 16, 'type': 'GetImageSize', 'pos': [1850, 0], 'size': [330, 270], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 17}], 'outputs': [{'name': 'width', 'type': 'INT', 'links': [18, 19]}, {'name': 'height', 'type': 'INT', 'links': [20, 21]}, {'name': 'batch_size', 'type': 'INT', 'links': []}], 'properties': {'Node name for S&R': 'GetImageSize'}, 'widgets_values': [], 'title': 'Use target image aspect ratio'}, {'id': 17, 'type': 'EmptyFlux2LatentImage', 'pos': [1850, 320], 'size': [330, 270], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 18}, {'name': 'height', 'type': 'INT', 'link': 20}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [29]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 18, 'type': 'Flux2Scheduler', 'pos': [1850, 640], 'size': [330, 270], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 19}, {'name': 'height', 'type': 'INT', 'link': 21}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [28]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 19, 'type': 'RandomNoise', 'pos': [2220, 0], 'size': [330, 270], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [25]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'SEED'}, {'id': 20, 'type': 'KSamplerSelect', 'pos': [2220, 320], 'size': [330, 270], 'flags': {}, 'order': 19, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [27]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 21, 'type': 'CFGGuider', 'pos': [2220, 640], 'size': [330, 270], 'flags': {}, 'order': 20, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 22}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 23}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 24}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [26]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 22, 'type': 'SamplerCustomAdvanced', 'pos': [2590, 0], 'size': [330, 270], 'flags': {}, 'order': 21, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 25}, {'name': 'guider', 'type': 'GUIDER', 'link': 26}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 27}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 28}, {'name': 'latent_image', 'type': 'LATENT', 'link': 29}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [30]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 23, 'type': 'VAEDecodeTiled', 'pos': [2590, 320], 'size': [330, 270], 'flags': {}, 'order': 22, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 30}, {'name': 'vae', 'type': 'VAE', 'link': 31}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [32]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 24, 'type': 'SaveImage', 'pos': [2590, 640], 'size': [330, 270], 'flags': {}, 'order': 23, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 32}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/FaceSwap'], 'title': 'SAVE IMAGE — original image, no lettering'}], 'links': [[1, 2, 0, 4, 0, 'CLIP'], [2, 4, 0, 5, 0, 'CONDITIONING'], [3, 6, 0, 8, 0, 'IMAGE'], [4, 8, 0, 9, 0, 'IMAGE'], [5, 3, 0, 9, 1, 'VAE'], [6, 4, 0, 10, 0, 'CONDITIONING'], [7, 9, 0, 10, 1, 'LATENT'], [8, 5, 0, 11, 0, 'CONDITIONING'], [9, 9, 0, 11, 1, 'LATENT'], [10, 7, 0, 12, 0, 'IMAGE'], [11, 12, 0, 13, 0, 'IMAGE'], [12, 3, 0, 13, 1, 'VAE'], [13, 10, 0, 14, 0, 'CONDITIONING'], [14, 13, 0, 14, 1, 'LATENT'], [15, 11, 0, 15, 0, 'CONDITIONING'], [16, 13, 0, 15, 1, 'LATENT'], [17, 8, 0, 16, 0, 'IMAGE'], [18, 16, 0, 17, 0, 'INT'], [19, 16, 0, 18, 0, 'INT'], [20, 16, 1, 17, 1, 'INT'], [21, 16, 1, 18, 1, 'INT'], [22, 1, 0, 21, 0, 'MODEL'], [23, 14, 0, 21, 1, 'CONDITIONING'], [24, 15, 0, 21, 2, 'CONDITIONING'], [25, 19, 0, 22, 0, 'NOISE'], [26, 21, 0, 22, 1, 'GUIDER'], [27, 20, 0, 22, 2, 'SAMPLER'], [28, 18, 0, 22, 3, 'SIGMAS'], [29, 17, 0, 22, 4, 'LATENT'], [30, 22, 0, 23, 0, 'LATENT'], [31, 3, 0, 23, 1, 'VAE'], [32, 23, 0, 24, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.75, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Face Swap - Masked': {'last_node_id': 25, 'last_link_id': 35, 'nodes': [{'id': 1, 'type': 'UNETLoader', 'pos': [0, 0], 'size': [330, 270], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [22]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 2, 'type': 'CLIPLoader', 'pos': [0, 320], 'size': [330, 270], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 3, 'type': 'VAELoader', 'pos': [0, 640], 'size': [330, 270], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [5, 12, 31]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 4, 'type': 'CLIPTextEncode', 'pos': [370, 0], 'size': [330, 270], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [2, 6]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['Use image 1 as the target photograph. Replace the selected person’s face identity with the person in image 2. Preserve the target pose, expression, clothing, framing and background.'], 'title': 'PROMPT — describe your image or edit'}, {'id': 5, 'type': 'ConditioningZeroOut', 'pos': [370, 320], 'size': [330, 270], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [8]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 6, 'type': 'LoadImage', 'pos': [370, 640], 'size': [330, 270], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3, 32]}, {'name': 'MASK', 'type': 'MASK', 'links': [34]}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'TARGET — image to edit / right-click → MaskEditor'}, {'id': 7, 'type': 'LoadImage', 'pos': [740, 0], 'size': [330, 270], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [10]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'SOURCE — face / identity reference'}, {'id': 8, 'type': 'ImageScaleToTotalPixels', 'pos': [740, 320], 'size': [330, 270], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 3}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4, 17]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 16], 'title': 'Reference size — about 1 megapixel'}, {'id': 9, 'type': 'VAEEncode', 'pos': [740, 640], 'size': [330, 270], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 4}, {'name': 'vae', 'type': 'VAE', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [7, 9]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 10, 'type': 'ReferenceLatent', 'pos': [1110, 0], 'size': [330, 270], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 6}, {'name': 'latent', 'type': 'LATENT', 'link': 7}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [13]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 11, 'type': 'ReferenceLatent', 'pos': [1110, 320], 'size': [330, 270], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 8}, {'name': 'latent', 'type': 'LATENT', 'link': 9}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [15]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 12, 'type': 'ImageScaleToTotalPixels', 'pos': [1110, 640], 'size': [330, 270], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 10}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [11]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 16], 'title': 'Reference size — about 1 megapixel'}, {'id': 13, 'type': 'VAEEncode', 'pos': [1480, 0], 'size': [330, 270], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 11}, {'name': 'vae', 'type': 'VAE', 'link': 12}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [14, 16]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 14, 'type': 'ReferenceLatent', 'pos': [1480, 320], 'size': [330, 270], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 13}, {'name': 'latent', 'type': 'LATENT', 'link': 14}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [23]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 15, 'type': 'ReferenceLatent', 'pos': [1480, 640], 'size': [330, 270], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 15}, {'name': 'latent', 'type': 'LATENT', 'link': 16}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [24]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 16, 'type': 'GetImageSize', 'pos': [1850, 0], 'size': [330, 270], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 17}], 'outputs': [{'name': 'width', 'type': 'INT', 'links': [18, 19]}, {'name': 'height', 'type': 'INT', 'links': [20, 21]}, {'name': 'batch_size', 'type': 'INT', 'links': []}], 'properties': {'Node name for S&R': 'GetImageSize'}, 'widgets_values': [], 'title': 'Use target image aspect ratio'}, {'id': 17, 'type': 'EmptyFlux2LatentImage', 'pos': [1850, 320], 'size': [330, 270], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 18}, {'name': 'height', 'type': 'INT', 'link': 20}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [29]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 18, 'type': 'Flux2Scheduler', 'pos': [1850, 640], 'size': [330, 270], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 19}, {'name': 'height', 'type': 'INT', 'link': 21}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [28]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 19, 'type': 'RandomNoise', 'pos': [2220, 0], 'size': [330, 270], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [25]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'SEED'}, {'id': 20, 'type': 'KSamplerSelect', 'pos': [2220, 320], 'size': [330, 270], 'flags': {}, 'order': 19, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [27]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 21, 'type': 'CFGGuider', 'pos': [2220, 640], 'size': [330, 270], 'flags': {}, 'order': 20, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 22}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 23}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 24}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [26]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 22, 'type': 'SamplerCustomAdvanced', 'pos': [2590, 0], 'size': [330, 270], 'flags': {}, 'order': 21, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 25}, {'name': 'guider', 'type': 'GUIDER', 'link': 26}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 27}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 28}, {'name': 'latent_image', 'type': 'LATENT', 'link': 29}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [30]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 23, 'type': 'VAEDecodeTiled', 'pos': [2590, 320], 'size': [330, 270], 'flags': {}, 'order': 22, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 30}, {'name': 'vae', 'type': 'VAE', 'link': 31}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [33]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 24, 'type': 'CommercialMaskedEdit', 'pos': [2590, 640], 'size': [330, 270], 'flags': {}, 'order': 23, 'mode': 0, 'inputs': [{'name': 'original', 'type': 'IMAGE', 'link': 32}, {'name': 'edited', 'type': 'IMAGE', 'link': 33}, {'name': 'mask', 'type': 'MASK', 'link': 34}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [35]}], 'properties': {'Node name for S&R': 'CommercialMaskedEdit'}, 'widgets_values': [12], 'title': 'Paste only inside painted mask'}, {'id': 25, 'type': 'SaveImage', 'pos': [2960, 0], 'size': [330, 270], 'flags': {}, 'order': 24, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 35}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/FaceSwap'], 'title': 'SAVE IMAGE — original image, no lettering'}], 'links': [[1, 2, 0, 4, 0, 'CLIP'], [2, 4, 0, 5, 0, 'CONDITIONING'], [3, 6, 0, 8, 0, 'IMAGE'], [4, 8, 0, 9, 0, 'IMAGE'], [5, 3, 0, 9, 1, 'VAE'], [6, 4, 0, 10, 0, 'CONDITIONING'], [7, 9, 0, 10, 1, 'LATENT'], [8, 5, 0, 11, 0, 'CONDITIONING'], [9, 9, 0, 11, 1, 'LATENT'], [10, 7, 0, 12, 0, 'IMAGE'], [11, 12, 0, 13, 0, 'IMAGE'], [12, 3, 0, 13, 1, 'VAE'], [13, 10, 0, 14, 0, 'CONDITIONING'], [14, 13, 0, 14, 1, 'LATENT'], [15, 11, 0, 15, 0, 'CONDITIONING'], [16, 13, 0, 15, 1, 'LATENT'], [17, 8, 0, 16, 0, 'IMAGE'], [18, 16, 0, 17, 0, 'INT'], [19, 16, 0, 18, 0, 'INT'], [20, 16, 1, 17, 1, 'INT'], [21, 16, 1, 18, 1, 'INT'], [22, 1, 0, 21, 0, 'MODEL'], [23, 14, 0, 21, 1, 'CONDITIONING'], [24, 15, 0, 21, 2, 'CONDITIONING'], [25, 19, 0, 22, 0, 'NOISE'], [26, 21, 0, 22, 1, 'GUIDER'], [27, 20, 0, 22, 2, 'SAMPLER'], [28, 18, 0, 22, 3, 'SIGMAS'], [29, 17, 0, 22, 4, 'LATENT'], [30, 22, 0, 23, 0, 'LATENT'], [31, 3, 0, 23, 1, 'VAE'], [32, 6, 0, 24, 0, 'IMAGE'], [33, 23, 0, 24, 1, 'IMAGE'], [34, 6, 1, 24, 2, 'MASK'], [35, 24, 0, 25, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.75, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Head Swap - BFS': {'last_node_id': 25, 'last_link_id': 33, 'nodes': [{'id': 1, 'type': 'UNETLoader', 'pos': [0, 0], 'size': [330, 270], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [22]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['flux-2-klein-4b.safetensors', 'default'], 'title': 'UNETLoader'}, {'id': 2, 'type': 'CLIPLoader', 'pos': [0, 320], 'size': [330, 270], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'CLIP', 'type': 'CLIP', 'links': [1]}], 'properties': {'Node name for S&R': 'CLIPLoader'}, 'widgets_values': ['qwen_3_4b.safetensors', 'flux2', 'default'], 'title': 'CLIPLoader'}, {'id': 3, 'type': 'VAELoader', 'pos': [0, 640], 'size': [330, 270], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [5, 12, 32]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['flux2-vae.safetensors'], 'title': 'VAELoader'}, {'id': 4, 'type': 'CLIPTextEncode', 'pos': [370, 0], 'size': [330, 270], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'clip', 'type': 'CLIP', 'link': 1}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [2, 6]}], 'properties': {'Node name for S&R': 'CLIPTextEncode'}, 'widgets_values': ['head_swap: Use Picture 1 as the base photograph. Replace its head with the head from Picture 2, preserving the reference identity, eyes, nose and hair. Match the target pose, gaze and expression. Keep body, clothing, framing, lighting and background intact.'], 'title': 'PROMPT — describe your image or edit'}, {'id': 5, 'type': 'ConditioningZeroOut', 'pos': [370, 320], 'size': [330, 270], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 2}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [8]}], 'properties': {'Node name for S&R': 'ConditioningZeroOut'}, 'widgets_values': [], 'title': 'ConditioningZeroOut'}, {'id': 6, 'type': 'LoadImage', 'pos': [370, 640], 'size': [330, 270], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'TARGET — image to edit'}, {'id': 7, 'type': 'LoadImage', 'pos': [740, 0], 'size': [330, 270], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [10]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'SOURCE — face / identity reference'}, {'id': 8, 'type': 'ImageScaleToTotalPixels', 'pos': [740, 320], 'size': [330, 270], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 3}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4, 17]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 16], 'title': 'Reference size — about 1 megapixel'}, {'id': 9, 'type': 'VAEEncode', 'pos': [740, 640], 'size': [330, 270], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 4}, {'name': 'vae', 'type': 'VAE', 'link': 5}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [7, 9]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 10, 'type': 'ReferenceLatent', 'pos': [1110, 0], 'size': [330, 270], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 6}, {'name': 'latent', 'type': 'LATENT', 'link': 7}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [13]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 11, 'type': 'ReferenceLatent', 'pos': [1110, 320], 'size': [330, 270], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 8}, {'name': 'latent', 'type': 'LATENT', 'link': 9}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [15]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 12, 'type': 'ImageScaleToTotalPixels', 'pos': [1110, 640], 'size': [330, 270], 'flags': {}, 'order': 11, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 10}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [11]}], 'properties': {'Node name for S&R': 'ImageScaleToTotalPixels'}, 'widgets_values': ['lanczos', 1.0, 16], 'title': 'Reference size — about 1 megapixel'}, {'id': 13, 'type': 'VAEEncode', 'pos': [1480, 0], 'size': [330, 270], 'flags': {}, 'order': 12, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 11}, {'name': 'vae', 'type': 'VAE', 'link': 12}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [14, 16]}], 'properties': {'Node name for S&R': 'VAEEncode'}, 'widgets_values': [], 'title': 'VAEEncode'}, {'id': 14, 'type': 'ReferenceLatent', 'pos': [1480, 320], 'size': [330, 270], 'flags': {}, 'order': 13, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 13}, {'name': 'latent', 'type': 'LATENT', 'link': 14}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [24]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 15, 'type': 'ReferenceLatent', 'pos': [1480, 640], 'size': [330, 270], 'flags': {}, 'order': 14, 'mode': 0, 'inputs': [{'name': 'conditioning', 'type': 'CONDITIONING', 'link': 15}, {'name': 'latent', 'type': 'LATENT', 'link': 16}], 'outputs': [{'name': 'CONDITIONING', 'type': 'CONDITIONING', 'links': [25]}], 'properties': {'Node name for S&R': 'ReferenceLatent'}, 'widgets_values': [], 'title': 'ReferenceLatent'}, {'id': 16, 'type': 'GetImageSize', 'pos': [1850, 0], 'size': [330, 270], 'flags': {}, 'order': 15, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 17}], 'outputs': [{'name': 'width', 'type': 'INT', 'links': [18, 19]}, {'name': 'height', 'type': 'INT', 'links': [20, 21]}, {'name': 'batch_size', 'type': 'INT', 'links': []}], 'properties': {'Node name for S&R': 'GetImageSize'}, 'widgets_values': [], 'title': 'Use target image aspect ratio'}, {'id': 17, 'type': 'EmptyFlux2LatentImage', 'pos': [1850, 320], 'size': [330, 270], 'flags': {}, 'order': 16, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 18}, {'name': 'height', 'type': 'INT', 'link': 20}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [30]}], 'properties': {'Node name for S&R': 'EmptyFlux2LatentImage'}, 'widgets_values': [1], 'title': 'EmptyFlux2LatentImage'}, {'id': 18, 'type': 'Flux2Scheduler', 'pos': [1850, 640], 'size': [330, 270], 'flags': {}, 'order': 17, 'mode': 0, 'inputs': [{'name': 'width', 'type': 'INT', 'link': 19}, {'name': 'height', 'type': 'INT', 'link': 21}], 'outputs': [{'name': 'SIGMAS', 'type': 'SIGMAS', 'links': [29]}], 'properties': {'Node name for S&R': 'Flux2Scheduler'}, 'widgets_values': [4], 'title': 'Flux2Scheduler'}, {'id': 19, 'type': 'LoraLoaderModelOnly', 'pos': [2220, 0], 'size': [330, 270], 'flags': {}, 'order': 18, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 22}], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [23]}], 'properties': {'Node name for S&R': 'LoraLoaderModelOnly'}, 'widgets_values': ['bfs_head_v1_flux-klein_4b.safetensors', 1.0], 'title': 'BFS head swap — strength 1.0'}, {'id': 20, 'type': 'RandomNoise', 'pos': [2220, 320], 'size': [330, 270], 'flags': {}, 'order': 19, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'NOISE', 'type': 'NOISE', 'links': [26]}], 'properties': {'Node name for S&R': 'RandomNoise'}, 'widgets_values': [42, 'randomize'], 'title': 'SEED'}, {'id': 21, 'type': 'KSamplerSelect', 'pos': [2220, 640], 'size': [330, 270], 'flags': {}, 'order': 20, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'SAMPLER', 'type': 'SAMPLER', 'links': [28]}], 'properties': {'Node name for S&R': 'KSamplerSelect'}, 'widgets_values': ['euler'], 'title': 'KSamplerSelect'}, {'id': 22, 'type': 'CFGGuider', 'pos': [2590, 0], 'size': [330, 270], 'flags': {}, 'order': 21, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 23}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 24}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 25}], 'outputs': [{'name': 'GUIDER', 'type': 'GUIDER', 'links': [27]}], 'properties': {'Node name for S&R': 'CFGGuider'}, 'widgets_values': [1.0], 'title': 'CFGGuider'}, {'id': 23, 'type': 'SamplerCustomAdvanced', 'pos': [2590, 320], 'size': [330, 270], 'flags': {}, 'order': 22, 'mode': 0, 'inputs': [{'name': 'noise', 'type': 'NOISE', 'link': 26}, {'name': 'guider', 'type': 'GUIDER', 'link': 27}, {'name': 'sampler', 'type': 'SAMPLER', 'link': 28}, {'name': 'sigmas', 'type': 'SIGMAS', 'link': 29}, {'name': 'latent_image', 'type': 'LATENT', 'link': 30}], 'outputs': [{'name': 'output', 'type': 'LATENT', 'links': [31]}, {'name': 'denoised_output', 'type': 'LATENT', 'links': []}], 'properties': {'Node name for S&R': 'SamplerCustomAdvanced'}, 'widgets_values': [], 'title': 'SamplerCustomAdvanced'}, {'id': 24, 'type': 'VAEDecodeTiled', 'pos': [2590, 640], 'size': [330, 270], 'flags': {}, 'order': 23, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 31}, {'name': 'vae', 'type': 'VAE', 'link': 32}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [33]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 64, 64, 8], 'title': 'VAEDecodeTiled'}, {'id': 25, 'type': 'SaveImage', 'pos': [2960, 0], 'size': [330, 270], 'flags': {}, 'order': 24, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 33}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/HeadSwap'], 'title': 'SAVE IMAGE — original image, no lettering'}], 'links': [[1, 2, 0, 4, 0, 'CLIP'], [2, 4, 0, 5, 0, 'CONDITIONING'], [3, 6, 0, 8, 0, 'IMAGE'], [4, 8, 0, 9, 0, 'IMAGE'], [5, 3, 0, 9, 1, 'VAE'], [6, 4, 0, 10, 0, 'CONDITIONING'], [7, 9, 0, 10, 1, 'LATENT'], [8, 5, 0, 11, 0, 'CONDITIONING'], [9, 9, 0, 11, 1, 'LATENT'], [10, 7, 0, 12, 0, 'IMAGE'], [11, 12, 0, 13, 0, 'IMAGE'], [12, 3, 0, 13, 1, 'VAE'], [13, 10, 0, 14, 0, 'CONDITIONING'], [14, 13, 0, 14, 1, 'LATENT'], [15, 11, 0, 15, 0, 'CONDITIONING'], [16, 13, 0, 15, 1, 'LATENT'], [17, 8, 0, 16, 0, 'IMAGE'], [18, 16, 0, 17, 0, 'INT'], [19, 16, 0, 18, 0, 'INT'], [20, 16, 1, 17, 1, 'INT'], [21, 16, 1, 18, 1, 'INT'], [22, 1, 0, 19, 0, 'MODEL'], [23, 19, 0, 22, 0, 'MODEL'], [24, 14, 0, 22, 1, 'CONDITIONING'], [25, 15, 0, 22, 2, 'CONDITIONING'], [26, 20, 0, 23, 0, 'NOISE'], [27, 22, 0, 23, 1, 'GUIDER'], [28, 21, 0, 23, 2, 'SAMPLER'], [29, 18, 0, 23, 3, 'SIGMAS'], [30, 17, 0, 23, 4, 'LATENT'], [31, 23, 0, 24, 0, 'LATENT'], [32, 3, 0, 24, 1, 'VAE'], [33, 24, 0, 25, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.75, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Face Composite - Simple': {'last_node_id': 5, 'last_link_id': 4, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 0], 'size': [330, 270], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [1]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'SOURCE — face to transfer'}, {'id': 2, 'type': 'LoadImage', 'pos': [0, 320], 'size': [330, 270], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [2]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'TARGET — image to change'}, {'id': 3, 'type': 'CommercialMeshSwap', 'pos': [0, 640], 'size': [330, 270], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'source', 'type': 'IMAGE', 'link': 1}, {'name': 'target', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'image', 'type': 'IMAGE', 'links': [3]}, {'name': 'report', 'type': 'STRING', 'links': [4]}], 'properties': {'Node name for S&R': 'CommercialMeshSwap'}, 'widgets_values': [0, 0, 'Feather'], 'title': 'Face composite — similar poses only'}, {'id': 4, 'type': 'SaveImage', 'pos': [370, 0], 'size': [330, 270], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 3}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/FaceComposite'], 'title': 'SaveImage'}, {'id': 5, 'type': 'PreviewAny', 'pos': [370, 320], 'size': [330, 270], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'source', 'type': '*', 'link': 4}], 'outputs': [{'name': 'STRING', 'type': 'STRING', 'links': []}], 'properties': {'Node name for S&R': 'PreviewAny'}, 'widgets_values': [], 'title': 'PreviewAny'}], 'links': [[1, 1, 0, 3, 0, 'IMAGE'], [2, 2, 0, 3, 1, 'IMAGE'], [3, 3, 0, 4, 0, 'IMAGE'], [4, 3, 1, 5, 0, 'STRING']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.75, 'offset': [150, 150]}, 'model_profile': 'commercial-v7'}, 'version': 0.4}, 'Upscale 2x': {'last_node_id': 5, 'last_link_id': 4, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 100], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [2]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload an image to enlarge'}, {'id': 2, 'type': 'UpscaleModelLoader', 'pos': [0, 440], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'UPSCALE_MODEL', 'type': 'UPSCALE_MODEL', 'links': [1]}], 'properties': {'Node name for S&R': 'UpscaleModelLoader'}, 'widgets_values': ['RealESRGAN_x4plus.pth']}, {'id': 3, 'type': 'ImageUpscaleWithModel', 'pos': [360, 100], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'upscale_model', 'type': 'UPSCALE_MODEL', 'link': 1}, {'name': 'image', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}], 'properties': {'Node name for S&R': 'ImageUpscaleWithModel'}, 'widgets_values': []}, {'id': 4, 'type': 'ImageScaleBy', 'pos': [720, 100], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 3}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [4]}], 'properties': {'Node name for S&R': 'ImageScaleBy'}, 'widgets_values': ['lanczos', 0.5], 'title': '4x model then downsample to 2x'}, {'id': 5, 'type': 'SaveImage', 'pos': [1080, 100], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 4}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Upscale 2x']}], 'links': [[1, 2, 0, 3, 0, 'UPSCALE_MODEL'], [2, 1, 0, 3, 1, 'IMAGE'], [3, 3, 0, 4, 0, 'IMAGE'], [4, 4, 0, 5, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}}, 'version': 0.4}, 'Upscale 4x': {'last_node_id': 4, 'last_link_id': 3, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 100], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [2]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload an image to enlarge'}, {'id': 2, 'type': 'UpscaleModelLoader', 'pos': [0, 440], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'UPSCALE_MODEL', 'type': 'UPSCALE_MODEL', 'links': [1]}], 'properties': {'Node name for S&R': 'UpscaleModelLoader'}, 'widgets_values': ['RealESRGAN_x4plus.pth']}, {'id': 3, 'type': 'ImageUpscaleWithModel', 'pos': [360, 100], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'upscale_model', 'type': 'UPSCALE_MODEL', 'link': 1}, {'name': 'image', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [3]}], 'properties': {'Node name for S&R': 'ImageUpscaleWithModel'}, 'widgets_values': []}, {'id': 4, 'type': 'SaveImage', 'pos': [1080, 100], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 3}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Upscale 4x']}], 'links': [[1, 2, 0, 3, 0, 'UPSCALE_MODEL'], [2, 1, 0, 3, 1, 'IMAGE'], [3, 3, 0, 4, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}}, 'version': 0.4}, 'Restore SeedVR2': {'last_node_id': 11, 'last_link_id': 15, 'nodes': [{'id': 1, 'type': 'LoadImage', 'pos': [0, 100], 'size': [320, 250], 'flags': {}, 'order': 0, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [1]}, {'name': 'MASK', 'type': 'MASK', 'links': []}], 'properties': {'Node name for S&R': 'LoadImage'}, 'widgets_values': ['', 'image'], 'title': 'Upload an image to enlarge'}, {'id': 2, 'type': 'ImageScaleBy', 'pos': [360, 100], 'size': [320, 250], 'flags': {}, 'order': 1, 'mode': 0, 'inputs': [{'name': 'image', 'type': 'IMAGE', 'link': 1}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [2, 14]}], 'properties': {'Node name for S&R': 'ImageScaleBy'}, 'widgets_values': ['lanczos', 2.0], 'title': '2x target; start with source at 1024px or smaller'}, {'id': 3, 'type': 'SeedVR2Preprocess', 'pos': [720, 100], 'size': [320, 250], 'flags': {}, 'order': 2, 'mode': 0, 'inputs': [{'name': 'resized_images', 'type': 'IMAGE', 'link': 2}], 'outputs': [{'name': 'images', 'type': 'IMAGE', 'links': [3]}], 'properties': {'Node name for S&R': 'SeedVR2Preprocess'}, 'widgets_values': []}, {'id': 4, 'type': 'VAELoader', 'pos': [360, 440], 'size': [320, 250], 'flags': {}, 'order': 3, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'VAE', 'type': 'VAE', 'links': [4, 12]}], 'properties': {'Node name for S&R': 'VAELoader'}, 'widgets_values': ['seedvr2_ema_vae_fp16.safetensors']}, {'id': 5, 'type': 'UNETLoader', 'pos': [0, 440], 'size': [320, 250], 'flags': {}, 'order': 4, 'mode': 0, 'inputs': [], 'outputs': [{'name': 'MODEL', 'type': 'MODEL', 'links': [5, 7]}], 'properties': {'Node name for S&R': 'UNETLoader'}, 'widgets_values': ['seedvr2_3b_int8_convrot.safetensors', 'default']}, {'id': 6, 'type': 'VAEEncodeTiled', 'pos': [1080, 100], 'size': [320, 250], 'flags': {}, 'order': 5, 'mode': 0, 'inputs': [{'name': 'pixels', 'type': 'IMAGE', 'link': 3}, {'name': 'vae', 'type': 'VAE', 'link': 4}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [6, 10]}], 'properties': {'Node name for S&R': 'VAEEncodeTiled'}, 'widgets_values': [512, 128, 64, 8]}, {'id': 7, 'type': 'SeedVR2Conditioning', 'pos': [1080, 440], 'size': [320, 250], 'flags': {}, 'order': 6, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 5}, {'name': 'vae_conditioning', 'type': 'LATENT', 'link': 6}], 'outputs': [{'name': 'positive', 'type': 'CONDITIONING', 'links': [8]}, {'name': 'negative', 'type': 'CONDITIONING', 'links': [9]}], 'properties': {'Node name for S&R': 'SeedVR2Conditioning'}, 'widgets_values': []}, {'id': 8, 'type': 'KSampler', 'pos': [1440, 100], 'size': [320, 250], 'flags': {}, 'order': 7, 'mode': 0, 'inputs': [{'name': 'model', 'type': 'MODEL', 'link': 7}, {'name': 'positive', 'type': 'CONDITIONING', 'link': 8}, {'name': 'negative', 'type': 'CONDITIONING', 'link': 9}, {'name': 'latent_image', 'type': 'LATENT', 'link': 10}], 'outputs': [{'name': 'LATENT', 'type': 'LATENT', 'links': [11]}], 'properties': {'Node name for S&R': 'KSampler'}, 'widgets_values': [42, 'fixed', 1, 1.0, 'euler', 'simple', 1.0]}, {'id': 9, 'type': 'VAEDecodeTiled', 'pos': [1800, 100], 'size': [320, 250], 'flags': {}, 'order': 8, 'mode': 0, 'inputs': [{'name': 'samples', 'type': 'LATENT', 'link': 11}, {'name': 'vae', 'type': 'VAE', 'link': 12}], 'outputs': [{'name': 'IMAGE', 'type': 'IMAGE', 'links': [13]}], 'properties': {'Node name for S&R': 'VAEDecodeTiled'}, 'widgets_values': [512, 128, 64, 8]}, {'id': 10, 'type': 'SeedVR2PostProcessing', 'pos': [2160, 100], 'size': [320, 250], 'flags': {}, 'order': 9, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 13}, {'name': 'original_resized_images', 'type': 'IMAGE', 'link': 14}], 'outputs': [{'name': 'images', 'type': 'IMAGE', 'links': [15]}], 'properties': {'Node name for S&R': 'SeedVR2PostProcessing'}, 'widgets_values': ['none']}, {'id': 11, 'type': 'SaveImage', 'pos': [2520, 100], 'size': [320, 250], 'flags': {}, 'order': 10, 'mode': 0, 'inputs': [{'name': 'images', 'type': 'IMAGE', 'link': 15}], 'outputs': [], 'properties': {'Node name for S&R': 'SaveImage'}, 'widgets_values': ['ImageStudio/Restore SeedVR2']}], 'links': [[1, 1, 0, 2, 0, 'IMAGE'], [2, 2, 0, 3, 0, 'IMAGE'], [3, 3, 0, 6, 0, 'IMAGE'], [4, 4, 0, 6, 1, 'VAE'], [5, 5, 0, 7, 0, 'MODEL'], [6, 6, 0, 7, 1, 'LATENT'], [7, 5, 0, 8, 0, 'MODEL'], [8, 7, 0, 8, 1, 'CONDITIONING'], [9, 7, 1, 8, 2, 'CONDITIONING'], [10, 6, 0, 8, 3, 'LATENT'], [11, 8, 0, 9, 0, 'LATENT'], [12, 4, 0, 9, 1, 'VAE'], [13, 9, 0, 10, 0, 'IMAGE'], [14, 2, 0, 10, 1, 'IMAGE'], [15, 10, 0, 11, 0, 'IMAGE']], 'groups': [], 'config': {}, 'extra': {'ds': {'scale': 0.65, 'offset': [180, 180]}}, 'version': 0.4}}
COMFY_REV='1332fabfcc507aa7c548cd5152071ed494c207e8'
CF_SHA='1f99fb518b83dec2ba8d870a7cc991d82a77e6196295a3d2754ba17ab9f91872'
# Isolate reviewed code/models from historical research-only installations.
BASE=Path('/kaggle/temp/image-studio-commercial-v7'); BASE.mkdir(parents=True,exist_ok=True)
OUT=Path('/kaggle/working/image-studio'); OUT.mkdir(parents=True,exist_ok=True)
lock=open(BASE/'startup.lock','w')
try: fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
except BlockingIOError: raise RuntimeError('Image Studio is already running. Stop its startup cell before starting again.')
children=[]
# Fail before downloading if the previous studio still owns this port.
import socket
probe=socket.socket()
try: probe.bind(('127.0.0.1',8188))
except OSError: raise RuntimeError('Port 8188 is occupied. Stop the earlier studio cell before starting Normal v10.')
finally: probe.close()

def run(args,**kw):subprocess.run(list(map(str,args)),check=True,**kw)
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
 return h.hexdigest()
def download(url,path,expected,size=None):
 if path.exists() and sha(path)==expected:return
 if path.exists():raise RuntimeError(f'Checksum mismatch: {path}. Preserve or move this file before retrying.')
 if size and shutil.disk_usage(path.parent).free<size+2*1024**3:raise RuntimeError('Insufficient free disk for model plus 2 GiB reserve.')
 temp=path.with_suffix(path.suffix+'.partial')
 print('Downloading',path.name,flush=True)
 with urllib.request.urlopen(url,timeout=120) as response,open(temp,'wb') as target:shutil.copyfileobj(response,target,8*1024*1024)
 if sha(temp)!=expected:raise RuntimeError(f'Download checksum mismatch: {temp}')
 temp.replace(path)
def spawn(args,log):
 f=open(log,'w');p=subprocess.Popen(list(map(str,args)),stdout=f,stderr=subprocess.STDOUT,start_new_session=True);f.close();children.append(p);return p
# Only allowlisted weights can enter this fresh installation. Never reuse old plugins.
assets=[a for a in CATALOG['assets'] if (a['pack']!='create_edit' or ENABLE_CREATE_EDIT) and (not a['name'].startswith('seedvr2') or ENABLE_SEEDVR2) and (not a['name'].endswith('.task') or ENABLE_MESH_SWAP) and (a['folder']!='loras' or (ENABLE_CREATE_EDIT and ENABLE_BFS_HEADSWAP))]
try:
 resolved_models=prepare_models(assets,
  [Path('/kaggle/input'),Path('/kaggle/working/image-studio-model-packs'),BASE/'ComfyUI/models',Path('/kaggle/temp/image-studio-commercial-v7/ComfyUI/models')])
except BaseException:
 fcntl.flock(lock,fcntl.LOCK_UN);lock.close();raise
try:
 import torch
 if not torch.cuda.is_available():raise RuntimeError('Select a Kaggle GPU accelerator and restart this cell.')
 print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__,flush=True)
 free,total=torch.cuda.mem_get_info(0)
 if free<10*1024**3:raise RuntimeError('GPU 0 has less than 10 GiB free. Stop other GPU notebooks/processes first.')
 # Separate T4s do not form one shared VRAM pool. This studio uses GPU 0.
 comfy=BASE/'ComfyUI'
 if not comfy.exists():
  run(['git','clone','--no-checkout','https://github.com/comfyanonymous/ComfyUI.git',comfy])
  run(['git','-C',comfy,'checkout','--detach',COMFY_REV])
 if subprocess.check_output(['git','-C',str(comfy),'rev-parse','HEAD'],text=True).strip()!=COMFY_REV:raise RuntimeError('Unexpected ComfyUI revision.')
 venv=BASE/'venv';python=venv/'bin/python'
 if not python.exists():run([sys.executable,'-m','venv','--system-site-packages','--without-pip',venv])
 constraints=BASE/'constraints.txt'
 import importlib.metadata as meta
 protected=['torch','torchvision','torchaudio','numpy','scipy']
 constraints.write_text('\n'.join(f'{p}=={meta.version(p)}' for p in protected))
 requirements=BASE/'requirements.txt'
 requirements.write_text('\n'.join(x for x in (comfy/'requirements.txt').read_text().splitlines() if x.strip() not in protected)+'\ntransformers==5.22.2\nhuggingface-hub==2.4.0\n')
 fingerprint=hashlib.sha256((requirements.read_text()+constraints.read_text()+sys.version).encode()).hexdigest()
 marker=BASE/'setup.complete'
 if not marker.exists() or marker.read_text()!=fingerprint:
  run([sys.executable,'-m','pip','--python',python,'install','-r',requirements,'-c',constraints])
  run([python,'-c','import torch, torchvision, transformers, aiohttp, safetensors; assert torch.cuda.is_available()'])
  marker.write_text(fingerprint)
 if ENABLE_MESH_SWAP:
  # Some minimal Linux images lack GLES, required even for MediaPipe CPU inference.
  import ctypes.util
  if not ctypes.util.find_library('GLESv2'):
   deb=BASE/'libgles2.deb'
   download('https://deb.debian.org/debian/pool/main/libg/libglvnd/libgles2_1.7.0-1+b2_amd64.deb',deb,'e12c18aa77939d122ee1dea520ba0440b97dc19200b7396e03dff03a3ece74c1')
   run(['dpkg-deb','-x',deb,BASE/'gles'])
   os.environ['LD_LIBRARY_PATH']=str(BASE/'gles/usr/lib/x86_64-linux-gnu')+':'+os.environ.get('LD_LIBRARY_PATH','')
  face_marker=BASE/'mediapipe-setup.complete'
  if not face_marker.exists() or face_marker.read_text()!=fingerprint:
   run([sys.executable,'-m','pip','--python',python,'install','mediapipe==1.2.2','-c',constraints])
   face_marker.write_text(fingerprint)
  run([python,'-c','import mediapipe as mp; mp.tasks.vision.FaceLandmarkerOptions'])
 custom=comfy/'custom_nodes/commercial_image_tools';custom.mkdir(parents=True,exist_ok=True)
 (custom/'__init__.py').write_text(FACE_NODE_SOURCE)
 allowed={'commercial_image_tools','jjp_image_production','__pycache__'}
 unexpected=[p.name for p in (comfy/'custom_nodes').iterdir() if p.is_dir() and p.name not in allowed and not p.name.startswith('.')]
 if unexpected:raise RuntimeError('Unexpected custom node directories in isolated commercial installation: '+', '.join(unexpected))
 run([sys.executable,'-m','pip','--python',python,'freeze'],stdout=open(OUT/'packages.txt','w'))
 for asset in assets:
  dest=comfy/'models'/asset['folder']/asset['name'];dest.parent.mkdir(parents=True,exist_ok=True)
  if dest.is_symlink() and not dest.exists():dest.unlink()
  if not dest.exists() and asset['name'] in resolved_models:dest.symlink_to(resolved_models[asset['name']])
  if not dest.exists():raise RuntimeError('Saved model disappeared: '+asset['name']+'. Reattach the ALL MODELS Dataset; no model download started.')
  if sha(dest)!=asset['sha256']:raise RuntimeError('Asset checksum mismatch: '+asset['name'])
 def workflow_enabled(wf):
  types={n['type'] for n in wf['nodes']}
  return (ENABLE_CREATE_EDIT or 'EmptyFlux2LatentImage' not in types) and (ENABLE_MESH_SWAP or 'CommercialMeshSwap' not in types) and (ENABLE_SEEDVR2 or 'SeedVR2Conditioning' not in types) and (ENABLE_BFS_HEADSWAP or 'LoraLoaderModelOnly' not in types)
 WORKFLOW_PAYLOAD={k:v for k,v in WORKFLOW_PAYLOAD.items() if workflow_enabled(v)}
 workflow_dir=comfy/'user/default/workflows';workflow_dir.mkdir(parents=True,exist_ok=True)
 install_workflows(workflow_dir,WORKFLOW_PAYLOAD,OUT/'previous-workflows')
 # Stable output destination: PNGs retain the actual graph, prompt, seed and settings.
 for name,content in LICENSE_FILES.items():(OUT/name).write_text(content)
 (OUT/'provenance.json').write_text(json.dumps({'profile':'commercial-v7','comfy_revision':COMFY_REV,'assets':assets,'torch':torch.__version__},indent=2))
 cf=BASE/'cloudflared'
 download('https://github.com/cloudflare/cloudflared/releases/download/2026.10.7/cloudflared-linux-amd64',cf,CF_SHA)
 cf.chmod(0o700)
 server=spawn([python,comfy/'main.py','--listen','127.0.0.1','--port','8188','--lowvram','--force-fp16','--fp16-text-enc','--preview-method','none','--output-directory',OUT/'images','--disable-auto-launch','--disable-all-custom-nodes','--whitelist-custom-nodes','commercial_image_tools'],OUT/'comfy.log')
 deadline=time.monotonic()+180
 while True:
  if server.poll() is not None:raise RuntimeError('ComfyUI stopped. See comfy.log in Kaggle outputs.')
  try:
   with urllib.request.urlopen('http://127.0.0.1:8188/object_info',timeout=3) as r:info=json.load(r)
   break
  except OSError:
   if time.monotonic()>deadline:raise RuntimeError('ComfyUI startup timed out. See comfy.log.')
   time.sleep(1)
 # Validate installed node classes before offering the browser link.
 for wf in WORKFLOW_PAYLOAD.values():
  for node in wf['nodes']:
   if node['type'] not in info:raise RuntimeError('Required node unavailable: '+node['type'])
 tunnel=spawn([cf,'tunnel','--url','http://127.0.0.1:8188','--no-autoupdate'],OUT/'tunnel.log')
 deadline=time.monotonic()+90
 url=None
 while time.monotonic()<deadline:
  text=(OUT/'tunnel.log').read_text()
  matches=re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
  if matches:url=matches[-1];break
  if tunnel.poll() is not None:raise RuntimeError('Cloudflare stopped; inspect tunnel.log.')
  time.sleep(1)
 if not url:raise RuntimeError('No Cloudflare URL received; inspect tunnel.log.')
 print('\nOPEN IMAGE STUDIO:',url,flush=True)
 print('Workflow menu → Image Studio: '+', '.join(WORKFLOW_PAYLOAD)+'. Select a workflow, then press Run.')
 print('No generation has started. Keep this cell running. Anyone with this session URL can access the studio.')
 print('Save Version with outputs before ending Kaggle. Images:',OUT/'images',flush=True)
 while server.poll() is None and tunnel.poll() is None:time.sleep(2)
 raise RuntimeError('Studio or tunnel stopped; see the saved logs.')
finally:
 for p in reversed(children):
  if p.poll() is None:
   os.killpg(p.pid,signal.SIGTERM)
   try:p.wait(timeout=10)
   except subprocess.TimeoutExpired:os.killpg(p.pid,signal.SIGKILL);p.wait()
 fcntl.flock(lock,fcntl.LOCK_UN);lock.close()
